Load packages

In [5]:
import os
import napari
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
import tifffile as tiff
import cv2
from collections import defaultdict
from skimage import morphology
from scipy import ndimage as ndi
from skimage import filters, morphology, measure, segmentation, feature
from skimage.transform import resize
import re
import shutil
from scipy.ndimage import binary_fill_holes 
from sam2.build_sam import build_sam2_video_predictor
from pathlib import Path

# napari 라이브러리는 별도로 필요할 때만 import합니다
# (Jupyter 환경에서 GUI 라이브러리 초기화가 커널 충돌을 유발할 수 있음)

Select device for computation

In [2]:
# select the device for computation
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"using device: {device}")

if device.type == "cuda":

    torch.autocast("cuda", dtype=torch.bfloat16).__enter__()

    if torch.cuda.get_device_properties(0).major >= 8:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
elif device.type == "mps":
    print(
        "\nSupport for MPS devices is preliminary. SAM 2 is trained with CUDA and might "
        "give numerically different outputs and sometimes degraded performance on MPS. "
        "See e.g. https://github.com/pytorch/pytorch/issues/84936 for a discussion."
    )

using device: cuda


Load SAM2

In [3]:
# directory for SAM2 packages
sam2_pkg_dir = "C:\Windows\System32\segment-anything-2\sam2"

# save directory for images to be processed
base_dir = "E:\python projects\codes\sam2"

# initiate SAM2 
# directory for checkpoints of SAM2
sam2_checkpoint = os.path.join(base_dir, "checkpoints", "sam2.1_hiera_small.pt")

# directory for config of SAM2 
model_cfg = os.path.join(base_dir,"sam2", "configs", "sam2.1", "sam2.1_hiera_s.yaml")

# generate predictor 
predictor = build_sam2_video_predictor(model_cfg, sam2_checkpoint, device=device)

In [4]:
import inspect, pydoc

cls = type(predictor)
print("Class:", cls.__module__, cls.__name__)

# 1) __init__ 시그니처
try:
    sig = inspect.signature(cls.__init__)
    print("\n[__init__ signature]")
    print(sig)
except Exception as e:
    print("\n[__init__ signature] failed:", e)

# 2) __init__ 문서 문자열
doc = getattr(cls.__init__, "__doc__", None)
if doc:
    print("\n[__init__ docstring]\n", doc)

# 3) 소스 코드 경로/일부
try:
    src = inspect.getsource(cls.__init__)
    print("\n[__init__ source snippet]\n", src[:2000])  # 길면 앞부분만
except Exception as e:
    try:
        fpath = inspect.getfile(cls)
        print("\n[Source file path]", fpath)
        # 필요하면 파일 열어서 'show'/'visual'/'viewer' 문자열 검색
    except Exception as e2:
        print("\n[Source path lookup failed]", e, e2)


Class: sam2.sam2_video_predictor SAM2VideoPredictor

[__init__ signature]
(self, fill_hole_area=0, non_overlap_masks=False, clear_non_cond_mem_around_input=False, add_all_frames_to_correct_as_cond=False, **kwargs)

[__init__ source snippet]
     def __init__(
        self,
        fill_hole_area=0,
        # whether to apply non-overlapping constraints on the output object masks
        non_overlap_masks=False,
        # whether to clear non-conditioning memory of the surrounding frames (which may contain outdated information) after adding correction clicks;
        # note that this would only apply to *single-object tracking* unless `clear_non_cond_mem_for_multi_obj` is also set to True)
        clear_non_cond_mem_around_input=False,
        # if `add_all_frames_to_correct_as_cond` is True, we also append to the conditioning frame list any frame that receives a later correction click
        # if `add_all_frames_to_correct_as_cond` is False, we conditioning frame list to only use thos

In [6]:

# --- Pre-process  
def preprocess(file_path):
    """
    TIFF 3D 스택을 읽어 다운샘플/등방성 근사 → 3D Sobel 경계강도 → 정규화 → 축 변환.
    반환:
        pre_data : 최종 전처리된 그라디언트 볼륨 (Z,Y,X)
        data     : 리사이즈 및 축 변환된 원본 볼륨 (Z,Y,X)
    """
    # 0) load
    
    data_name = os.path.basename(file_path.rstrip("/\\"))  # → 'cc'
    data_path = os.path.join(file_path)

    data = tiff.imread(data_path)
    data = np.array(data)

    if data.ndim != 3:
        raise ValueError(f"3D TIFF가 필요합니다. 현재 shape={data.shape}")

    # 1) Resizing: downsampled isotropic resolution (원 코드 그대로)
    dx = 0.1126   # in-plane (x,y) spacing
    dz = 0.5485   # slice spacing (z)

    W1 = data.shape[1]
    H1 = data.shape[2]
    D1 = int(round(data.shape[0] * dz / dx))

    W2 = int(round(0.3 * W1))
    H2 = int(round(0.3 * H1))
    D2 = int(round(0.3 * D1))

    data_size = (D2, W2, H2)  # 원 코드 유지
    data = resize(
        data, data_size, order=1, mode='edge', anti_aliasing=False, preserve_range=True
    ).astype(data.dtype, copy=False)

    # 2) 3D gradient (Sobel magnitude) — 원 축 정의 유지
    def imgradient3(V):
        Gx = ndi.sobel(V, axis=1, mode='nearest')  # X(열)
        Gy = ndi.sobel(V, axis=0, mode='nearest')  # Y(행)
        Gz = ndi.sobel(V, axis=2, mode='nearest')  # Z(슬라이스)
        return np.sqrt(Gx*Gx + Gy*Gy + Gz*Gz)

    grad = imgradient3(data)

    # 3) normalize (원 코드 그대로)
    norm_min = np.max(grad) * 0.01
    norm_max = np.max(grad) * 0.2
    grad = np.clip(grad, norm_min, norm_max)
    grad = (grad - norm_min) / (norm_max - norm_min)   # ← normalize to [0,1]

    # 4) permute (원 주석 유지: (X, Y, Z) -> (Z, Y, X))
    data = np.transpose(data, (2, 0, 1))
    grad = np.transpose(grad, (2, 0, 1))

    pre_data = grad
    return pre_data, data   # pre_data : preprocessed HT data; data: HT data

# --- Prepare Manual Slice Labels
def load_labels(file_path, viewer=None):

    if viewer is None:
        try:
            viewer = napari.current_viewer()
        except Exception:
            viewer = napari.Viewer()

    # 'labels'가 이름에 포함된 파일만 선별
    candidates = [f for f in os.listdir(file_path) if "labels" in f.lower()]
    if not candidates:
            viewer.add_labels(
                np.zeros_like(viewer.layers[0].data, dtype=np.uint8),
                name="Labels"
            )

    for fname in sorted(candidates):
        fpath = os.path.join(file_path, fname)
        lower = fname.lower()

        try:
            if lower.endswith(('.tif', '.tiff')):
                arr = tiff.imread(fpath)
            elif lower.endswith('.npy'):
                arr = np.load(fpath)
            else:
                print(f"⏭️ Skip (unsupported): {fname}")
                continue
            if not np.issubdtype(arr.dtype, np.integer):
                arr = arr.astype(np.int32)          
            # 🔻 확장자 제거한 이름 사용
            layer_name = Path(fname).stem
            viewer.add_labels(arr, name=layer_name)
            print(f"✅ Loaded to napari: {layer_name} (from '{fname}', shape={arr.shape})")

        except Exception as e:
            print(f"❌ Failed to load '{fname}': {e}")

    return viewer

# --- Prepare jpg images for SAM2 operation 
def jpg_for_sam2(file_path, pre_data):

    prefix = os.path.splitext(os.path.basename(file_path))[0]
    # output_dir = os.path.join(file_path, prefix)
    output_dir = r"G:\RealData\sam2"

    os.makedirs(output_dir, exist_ok=True)    

    for i, frame in enumerate(pre_data, start=1):
        norm_frame = cv2.normalize(frame, None, 0, 255, cv2.NORM_MINMAX)
        uint8_frame = norm_frame.astype(np.uint8)
        rgb_frame = cv2.cvtColor(uint8_frame, cv2.COLOR_GRAY2BGR)
        filename = f"{i:04d}.jpg"
        out_path = os.path.join(output_dir, filename)
        cv2.imwrite(out_path, rgb_frame)

    print(f"✔ {len(pre_data)} slices saved to: {output_dir}")

    # viewer.add_image(pre_data, name='raw', rgb=False)
    
    # napari.run()

    return pre_data, output_dir

# --- SAM2 operation 
def propagate_mask_and_visualize(
    jpg_path,
    predictor,
    label_layer=None,
    obj_id=1,
    seed_planes=None,
    viewer=None
):
    import numpy as np, os, napari
    from napari.layers import Labels as LabelsLayer

    # (0) viewer
    if viewer is None:
        try:
            viewer = napari.current_viewer()
        except Exception:
            viewer = napari.Viewer()

    # (1) label_layer 자동
    if (label_layer is None) or (not hasattr(label_layer, "data")):
        try:
            label_layer = viewer.layers['Labels']
        except KeyError:
            candidates = [lyr for lyr in viewer.layers if isinstance(lyr, LabelsLayer)]
            if not candidates:
                raise RuntimeError(
                    "napari 뷰어에서 Labels 레이어를 찾지 못했습니다. "
                    "viewer.layers['Labels'] 이름을 맞추거나, Labels 레이어를 추가하세요."
                )
            label_layer = candidates[0]
            print(f"⚠️ 'Labels'라는 이름의 레이어가 없어 첫 번째 Labels 레이어('{label_layer.name}')를 사용합니다.")

    # (2) 데이터
    import numpy as np, os
    label_data = np.asarray(label_layer.data)
    if label_data.ndim != 3:
        raise ValueError(f"Labels 데이터는 (Z, Y, X) 3D여야 합니다. 현재 shape={label_data.shape}")
    num_slices = label_data.shape[0]

    # (A) obj_id 존재 슬라이스
    if seed_planes is None:
        seed_planes = np.where(
            (label_data == obj_id).reshape(num_slices, -1).any(axis=1)
        )[0].tolist()

    if not seed_planes:
        uniq = np.unique(label_data)
        raise ValueError(
            f"[중단] obj_id={obj_id} 라벨이 없습니다. Labels 유니크 값(일부): {uniq[:20]}"
        )

    print(f"[seed check] obj_id={obj_id}, seed_planes (len={len(seed_planes)}): "
          f"{seed_planes[:20]}{' ...' if len(seed_planes)>20 else ''}")

    # (3) SAM2 predictor 초기화
    inference_state = predictor.init_state(video_path=jpg_path)
    predictor.reset_state(inference_state)

    # (B) 씨드 추가
    for fidx in seed_planes:
        manual_mask = (label_data[fidx] == obj_id)
        if manual_mask.any():
            predictor.add_new_mask(
                inference_state=inference_state,
                frame_idx=fidx,
                obj_id=obj_id,
                mask=manual_mask,
            )
            print(f"✔ Added seed @ z={fidx} (px={int(manual_mask.sum())})")
        else:
            print(f"⚠️ z={fidx}에 obj_id={obj_id} 마스크가 비어 있음 → 건너뜀")

    # (4) 전파
    center_seed = seed_planes[len(seed_planes)//2]
    print(f"🔁 Propagating from center z={center_seed}, seeds={len(seed_planes)}")

    video_segments = {}
    for rev in (False, True):
        for out_frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(
            inference_state, start_frame_idx=center_seed, reverse=rev
        ):
            per_obj_output_mask = {
                out_obj_id: (out_mask_logits[i] > 0.0).cpu().numpy()
                for i, out_obj_id in enumerate(out_obj_ids)
            }
            if out_frame_idx in video_segments:
                for oid, mask in per_obj_output_mask.items():
                    if oid in video_segments[out_frame_idx]:
                        video_segments[out_frame_idx][oid] |= mask
                    else:
                        video_segments[out_frame_idx][oid] = mask
            else:
                video_segments[out_frame_idx] = per_obj_output_mask

    if len(video_segments) == 0:
        raise RuntimeError("No masks were propagated. Check your seed mask or predictor output.")

    # (5) 라벨 스택 생성
    sample_mask = np.squeeze(next(iter(next(iter(video_segments.values())).values())))
    height, width = sample_mask.shape
    label_stack = np.zeros((num_slices, height, width), dtype=np.uint8)

    for fidx, per_obj_mask in video_segments.items():
        if obj_id in per_obj_mask:
            mask_i = np.squeeze(per_obj_mask[obj_id])
            label_stack[fidx][mask_i > 0] = obj_id

    # # (6) 작업 끝나면 jpg_path 폴더 삭제
    # try:
    #     shutil.rmtree(jpg_path)
    #     print(f"🧹 Removed temporary folder: {jpg_path}")
    # except Exception as e:
    #     print(f"⚠️ Failed to remove {jpg_path}: {e}")

    return label_stack, video_segments

# --- Post-process 
def finalize_mask_3d(label_stack,
                     obj_id=1,
                     suffix=None,                # 🆕 'ooplasm' / 'inner' / 'outer' / 기타
                     return_as_label_id=True):
    
    mask = (label_stack == obj_id)
    if not mask.any():
        raise ValueError(f" obj_id={obj_id} 마스크가 비어 있습니다.")

    # --- preserve the largest object using watershed ---
    dist = ndi.distance_transform_edt(mask)
    thr = 0.3 * dist.max()
    core = dist > thr                      # seed mask
    markers, _ = ndi.label(core)           # 여러 seed 자동 분리
    lbl = segmentation.watershed(-dist, markers, mask=mask)
    mask = lbl == np.bincount(lbl.ravel())[1:].argmax()+1   

    # --- morphological operation (varies with target)
    s = str(suffix).lower()

    config = {
        'pb':         (3, ('opening', 'closing')),
        'ooplasm':    (5, ('opening', 'closing')),
    }
    size, ops = config.get(s, (5, ('closing', 'opening'))) # default
    print(f"[morphology] suffix='{s}', size={size}, ops={ops}")

    selem = morphology.ball(size)
    for op in ops:
        mask = getattr(morphology, f"binary_{op}")(mask, selem)

    dist = ndi.distance_transform_edt(mask)
    dist_smooth = ndi.gaussian_filter(dist.astype(float), sigma=2)
    mask = dist_smooth > 1

    if return_as_label_id:
        out = np.zeros_like(mask, dtype=np.uint8)
        out[mask] = np.uint8(obj_id)
        return out
    else:
        return mask.astype(np.uint8)

# --- Run SAM2 segmentation and post-process for multiple label layers and build a combined 3D mask stack
def process_main(
    jpg_path,
    predictor,
    obj_id=1,
    viewer=None,
    layer_name_keyword="Labels",   # 이 문자열을 포함한 napari Labels 레이어만 대상
    # --- pb 전처리용 외부 의존성 ---
    pre_data=None,                 # 원본/전처리할 볼륨 (Z,Y,X)
    file_path=None,                # jpg_for_sam2 에 전달할 원본 파일 경로
    jpg_for_sam2=None              # callable: (file_path, pre_data) -> (pre_data, output_dir)
):

    if viewer is None:
        try:
            viewer = napari.current_viewer()
        except Exception:
            viewer = napari.Viewer()

    # --- 대상 레이어 수집 ---
    layers = [lyr for lyr in viewer.layers
              if isinstance(lyr, LabelsLayer) and (layer_name_keyword.lower() in lyr.name.lower())]
    if not layers:
        raise RuntimeError(f"이름에 '{layer_name_keyword}'가 포함된 napari Labels 레이어를 찾지 못했습니다.")

    def extract_suffix(name, fallback_idx):
        # 예: 'Labels(inner)' -> 'inner', 'Labels-outer' -> 'outer'
        m = re.search(r'Labels\s*[\(\-_\s]*([^)]+)[\)]*', name, re.IGNORECASE)
        if m:
            suf = m.group(1).strip()
            suf = re.sub(r'\W+', '_', suf)
            if suf:
                return suf
        return f"labels{fallback_idx}"

    # --- 처리 순서: ooplasm -> inner -> outer -> pb (대소문자 무시) ---
    priority = {"ooplasm": 0, "inner": 1, "outer": 2, "pb": 3}
    indexed = []
    for idx, lyr in enumerate(layers, start=1):
        suf = extract_suffix(lyr.name, idx)
        prio = priority.get(suf.lower(), 999)  # 지정 외 항목은 맨 뒤
        indexed.append((prio, suf, lyr))
    indexed.sort(key=lambda x: (x[0], x[1].lower()))

    predicted_labels = {'per_layer': {}, 'order': []}

    for prio, suffix, lyr in indexed:
        suffix_norm = suffix.lower()
        print(f"\n===== Processing: '{lyr.name}' → suffix='{suffix_norm}' (prio={prio}) =====")
        # pb가 아닌 경우: 기존 파이프라인
        if suffix_norm != "pb":
            label_stack, _ = propagate_mask_and_visualize(
                jpg_path, predictor, lyr, obj_id, None, viewer
            )
            final_mask = finalize_mask_3d(label_stack, suffix=suffix)

        else:
            # pb 전처리 요건 확인
            have_inner = f"final_mask_inner" in globals()
            have_ooplas = f"final_mask_ooplasm" in globals()
            if not (have_inner and have_ooplas):
                print("⚠️ PB 전처리 불가: inner/ooplasm 결과가 필요합니다. PB는 건너뜁니다.")
                continue
            if (pre_data is None) or (file_path is None) or (jpg_for_sam2 is None):
                print("⚠️ PB 전처리 불가: pre_data/file_path/jpg_for_sam2 가 필요합니다. PB는 건너뜁니다.")
                continue

            # (inner & ~ooplasm) 마스크 계산
            inner_mask = (globals()["final_mask_inner"] > 0)
            ooplasm_mask = (globals()["final_mask_ooplasm"] > 0)
            pvs_mask = (inner_mask & ~ooplasm_mask)

            if pvs_mask.shape != pre_data.shape:
                raise ValueError(f"PB 전처리: core_mask shape {pvs_mask.shape} != pre_data shape {pre_data.shape}")

            # segmented pre_data 생성: 마스크 위치만 보존
            if np.issubdtype(pre_data.dtype, np.bool_):
                segmented_pre = pre_data & pvs_mask
            else:
                segmented_pre = (pre_data * pvs_mask.astype(pre_data.dtype))

            # jpg set 재생성
            _pre_data_for_sam2, pb_jpg_path = jpg_for_sam2(file_path, segmented_pre)

            # SAM2 실행 (pb용)
            label_stack, _ = propagate_mask_and_visualize(
                pb_jpg_path, predictor, lyr, obj_id, None, viewer
            )
            final_mask = finalize_mask_3d(label_stack, suffix=suffix)
            # viewer.add_labels(final_mask, name="mask_PB_raw")
            # viewer.add_image(segmented_pre, name="segmented_pre")

        # 개별 결과 저장/노출
        globals()[f"label_stack_{suffix_norm}"] = label_stack
        globals()[f"final_mask_{suffix_norm}"]  = final_mask
        predicted_labels['per_layer'][suffix_norm] = dict(label_stack=label_stack, final_mask=final_mask)
        predicted_labels['order'].append(suffix_norm)

    print("\n✅ 개별 마스크 생성 완료(처리 순서):", predicted_labels['order'])

    for k in list(globals().keys()):
        if k.startswith("final_mask_"):
            norm = k.lower()
            if norm not in globals():
                globals()[norm] = globals()[k]

    # --- 4개 마스크를 통합한 label stack 생성 ---
    need = ["ooplasm", "inner", "outer", "pb"]
    if all(f"final_mask_{n}" in globals() for n in need):
        ooplasm = (globals()["final_mask_ooplasm"] > 0)
        inner   = (globals()["final_mask_inner"]   > 0)
        outer   = (globals()["final_mask_outer"]   > 0)
        pb      = (globals()["final_mask_pb"]      > 0)

        label_stack_all = np.zeros_like(ooplasm, np.uint8)
        # 우선순위: ooplasm(1) → inner-only(2) → outer-only(3) → pb-only(4)
        label_stack_all[ooplasm] = 1
        label_stack_all[inner & ~ooplasm] = 2
        label_stack_all[outer & ~(ooplasm | inner)] = 3
        label_stack_all[pb] = 4

        globals()["label_stack_all"] = label_stack_all
        viewer.add_labels(label_stack_all, name="mask_ALL")
        print("✅ label_stack_all 생성 (1=ooplasm, 2=pvs, 3=zona, 4=pb)")
        predicted_labels["label_stack_all"] = label_stack_all
    else:
        missing = [n for n in need if f"final_mask_{n}" not in globals()]
        print(f"⚠️ 통합 레이블(label_stack_all) 생성 불가: 부족한 마스크 = {missing}")

    return predicted_labels



In [4]:
#multi compartment pre-process
# --- Pre-process  
def preprocess(file_path):
    """
    TIFF 3D 스택을 읽어 다운샘플/등방성 근사 → 3D Sobel 경계강도 → 정규화 → 축 변환.
    반환:
        pre_data : 최종 전처리된 그라디언트 볼륨 (Z,Y,X)
        data     : 리사이즈 및 축 변환된 원본 볼륨 (Z,Y,X)
    """
    # 0) load
    
    data_name = os.path.basename(file_path.rstrip("/\\"))  # → 'cc'
    data_path = os.path.join(file_path)

    data = tiff.imread(data_path)
    data = np.array(data)

    if data.ndim != 3:
        raise ValueError(f"3D TIFF가 필요합니다. 현재 shape={data.shape}")

    # 1) Resizing: downsampled isotropic resolution (원 코드 그대로)
    dx = 0.1126   # in-plane (x,y) spacing
    dz = 0.5485   # slice spacing (z)

    W1 = data.shape[1]
    H1 = data.shape[2]
    D1 = int(round(data.shape[0] * dz / dx))

    W2 = int(round(0.3 * W1))
    H2 = int(round(0.3 * H1))
    D2 = int(round(0.3 * D1))

    data_size = (D2, W2, H2)  # 원 코드 유지
    data = resize(
        data, data_size, order=1, mode='edge', anti_aliasing=False, preserve_range=True
    ).astype(data.dtype, copy=False)

    # 2) 3D gradient (Sobel magnitude) — 원 축 정의 유지
    def imgradient3(V):
        Gx = ndi.sobel(V, axis=1, mode='nearest')  # X(열)
        Gy = ndi.sobel(V, axis=0, mode='nearest')  # Y(행)
        Gz = ndi.sobel(V, axis=2, mode='nearest')  # Z(슬라이스)
        return np.sqrt(Gx*Gx + Gy*Gy + Gz*Gz)

    grad = imgradient3(data)

    # 3) normalize (원 코드 그대로)
    norm_min = np.max(grad) * 0.01
    norm_max = np.max(grad) * 0.2
    grad = np.clip(grad, norm_min, norm_max)
    grad = (grad - norm_min) / (norm_max - norm_min)   # ← normalize to [0,1]

    # 4) permute (원 주석 유지: (X, Y, Z) -> (Z, Y, X))
    data = np.transpose(data, (2, 0, 1))
    grad = np.transpose(grad, (2, 0, 1))

    pre_data = grad
    return pre_data, data   # pre_data : preprocessed HT data; data: HT data

# --- Prepare Manual Slice Labels
def load_labels(file_path, viewer=None):

    if viewer is None:
        try:
            viewer = napari.current_viewer()
        except Exception:
            viewer = napari.Viewer()

    # 'labels'가 이름에 포함된 파일만 선별
    candidates = [f for f in os.listdir(file_path) if "labels" in f.lower()]
    if not candidates:
            viewer.add_labels(
                np.zeros_like(viewer.layers[0].data, dtype=np.uint8),
                name="Labels"
            )

    for fname in sorted(candidates):
        fpath = os.path.join(file_path, fname)
        lower = fname.lower()

        try:
            if lower.endswith(('.tif', '.tiff')):
                arr = tiff.imread(fpath)
            elif lower.endswith('.npy'):
                arr = np.load(fpath)
            else:
                print(f"⏭️ Skip (unsupported): {fname}")
                continue
            if not np.issubdtype(arr.dtype, np.integer):
                arr = arr.astype(np.int32)          
            # 🔻 확장자 제거한 이름 사용
            layer_name = Path(fname).stem
            viewer.add_labels(arr, name=layer_name)
            print(f"✅ Loaded to napari: {layer_name} (from '{fname}', shape={arr.shape})")

        except Exception as e:
            print(f"❌ Failed to load '{fname}': {e}")

    return viewer

# --- Prepare jpg images for SAM2 operation 
def jpg_for_sam2(file_path, pre_data):

    prefix = os.path.splitext(os.path.basename(file_path))[0]
    # output_dir = os.path.join(file_path, prefix)
    output_dir = r"G:\RealData\sam2"

    os.makedirs(output_dir, exist_ok=True)    

    for i, frame in enumerate(pre_data, start=1):
        norm_frame = cv2.normalize(frame, None, 0, 255, cv2.NORM_MINMAX)
        uint8_frame = norm_frame.astype(np.uint8)
        rgb_frame = cv2.cvtColor(uint8_frame, cv2.COLOR_GRAY2BGR)
        filename = f"{i:04d}.jpg"
        out_path = os.path.join(output_dir, filename)
        cv2.imwrite(out_path, rgb_frame)

    print(f"✔ {len(pre_data)} slices saved to: {output_dir}")

    # viewer.add_image(pre_data, name='raw', rgb=False)
    
    # napari.run()

    return pre_data, output_dir

# --- SAM2 operation 
def propagate_mask_and_visualize(
    jpg_path,
    predictor,
    label_layer=None,
    obj_id=1,
    seed_planes=None,
    viewer=None
):
    import numpy as np, os, napari
    from napari.layers import Labels as LabelsLayer

    # (0) viewer
    if viewer is None:
        try:
            viewer = napari.current_viewer()
        except Exception:
            viewer = napari.Viewer()

    # (1) label_layer 자동
    if (label_layer is None) or (not hasattr(label_layer, "data")):
        try:
            label_layer = viewer.layers['Labels']
        except KeyError:
            candidates = [lyr for lyr in viewer.layers if isinstance(lyr, LabelsLayer)]
            if not candidates:
                raise RuntimeError(
                    "napari 뷰어에서 Labels 레이어를 찾지 못했습니다. "
                    "viewer.layers['Labels'] 이름을 맞추거나, Labels 레이어를 추가하세요."
                )
            label_layer = candidates[0]
            print(f"⚠️ 'Labels'라는 이름의 레이어가 없어 첫 번째 Labels 레이어('{label_layer.name}')를 사용합니다.")

    # (2) 데이터
    import numpy as np, os
    label_data = np.asarray(label_layer.data)
    if label_data.ndim != 3:
        raise ValueError(f"Labels 데이터는 (Z, Y, X) 3D여야 합니다. 현재 shape={label_data.shape}")
    num_slices = label_data.shape[0]

    # (A) obj_id 존재 슬라이스
    if seed_planes is None:
        seed_planes = np.where(
            (label_data >0 ).reshape(num_slices, -1).any(axis=1)
        )[0].tolist()

    if not seed_planes:
        uniq = np.unique(label_data)
        raise ValueError(
            f"[중단] nonzero 라벨이 없습니다. Labels 유니크 값(일부): {uniq[:20]}"
        )

    print(f"[seed check] nonzero seed, seed_planes (len={len(seed_planes)}): "
          f"{seed_planes[:20]}{' ...' if len(seed_planes)>20 else ''}")

    # (3) SAM2 predictor 초기화
    inference_state = predictor.init_state(video_path=jpg_path)
    predictor.reset_state(inference_state)

    # (B) 씨드 추가
    for fidx in seed_planes:
        manual_mask = (label_data[fidx] >0)
        if manual_mask.any():
            predictor.add_new_mask(
                inference_state=inference_state,
                frame_idx=fidx,
                obj_id=obj_id,
                mask=manual_mask,
            )
            print(f"✔ Added seed @ z={fidx} (px={int(manual_mask.sum())})")
        else:
            print(f"⚠️ z={fidx}에 obj_id={obj_id} 마스크가 비어 있음 → 건너뜀")

    # (4) 전파
    center_seed = seed_planes[len(seed_planes)//2]
    print(f"🔁 Propagating from center z={center_seed}, seeds={len(seed_planes)}")

    video_segments = {}
    for rev in (False, True):
        for out_frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(
            inference_state, start_frame_idx=center_seed, reverse=rev
        ):
            per_obj_output_mask = {
                out_obj_id: (out_mask_logits[i] > 0.0).cpu().numpy()
                for i, out_obj_id in enumerate(out_obj_ids)
            }
            if out_frame_idx in video_segments:
                for oid, mask in per_obj_output_mask.items():
                    if oid in video_segments[out_frame_idx]:
                        video_segments[out_frame_idx][oid] |= mask
                    else:
                        video_segments[out_frame_idx][oid] = mask
            else:
                video_segments[out_frame_idx] = per_obj_output_mask

    if len(video_segments) == 0:
        raise RuntimeError("No masks were propagated. Check your seed mask or predictor output.")

    # (5) 라벨 스택 생성
    sample_mask = np.squeeze(next(iter(next(iter(video_segments.values())).values())))
    height, width = sample_mask.shape
    label_stack = np.zeros((num_slices, height, width), dtype=np.uint8)

    for fidx, per_obj_mask in video_segments.items():
        if obj_id in per_obj_mask:
            mask_i = np.squeeze(per_obj_mask[obj_id])
            label_stack[fidx][mask_i > 0] = obj_id

    # # (6) 작업 끝나면 jpg_path 폴더 삭제
    # try:
    #     shutil.rmtree(jpg_path)
    #     print(f"🧹 Removed temporary folder: {jpg_path}")
    # except Exception as e:
    #     print(f"⚠️ Failed to remove {jpg_path}: {e}")

    return label_stack, video_segments

# --- Post-process 
def finalize_mask_3d(label_stack,
                     obj_id=1,
                     suffix=None,                # 🆕 'ooplasm' / 'inner' / 'outer' / 기타
                     return_as_label_id=True):
    
    mask = (label_stack == obj_id)
    if not mask.any():
        raise ValueError(f" obj_id={obj_id} 마스크가 비어 있습니다.")

    # --- preserve the largest object using watershed ---
    dist = ndi.distance_transform_edt(mask)
    thr = 0.3 * dist.max()
    core = dist > thr                      # seed mask
    markers, _ = ndi.label(core)           # 여러 seed 자동 분리
    lbl = segmentation.watershed(-dist, markers, mask=mask)
    mask = lbl == np.bincount(lbl.ravel())[1:].argmax()+1   

    # --- morphological operation (varies with target)
    s = str(suffix).lower()

    config = {
        'pb':         (3, ('opening', 'closing')),
        'ooplasm':    (5, ('opening', 'closing')),
    }
    size, ops = config.get(s, (5, ('closing', 'opening'))) # default
    print(f"[morphology] suffix='{s}', size={size}, ops={ops}")

    selem = morphology.ball(size)
    for op in ops:
        mask = getattr(morphology, f"binary_{op}")(mask, selem)

    dist = ndi.distance_transform_edt(mask)
    dist_smooth = ndi.gaussian_filter(dist.astype(float), sigma=2)
    mask = dist_smooth > 1

    if return_as_label_id:
        out = np.zeros_like(mask, dtype=np.uint8)
        out[mask] = np.uint8(obj_id)
        return out
    else:
        return mask.astype(np.uint8)

# --- Run SAM2 segmentation and post-process for multiple label layers and build a combined 3D mask stack
def process_main(
    jpg_path,
    predictor,
    obj_id=1,
    viewer=None,
    layer_name_keyword="Labels",   # 이 문자열을 포함한 napari Labels 레이어만 대상
    # --- pb 전처리용 외부 의존성 ---
    pre_data=None,                 # 원본/전처리할 볼륨 (Z,Y,X)
    file_path=None,                # jpg_for_sam2 에 전달할 원본 파일 경로
    jpg_for_sam2=None              # callable: (file_path, pre_data) -> (pre_data, output_dir)
):

    if viewer is None:
        try:
            viewer = napari.current_viewer()
        except Exception:
            viewer = napari.Viewer()

    # --- 대상 레이어 수집 ---
    layers = [lyr for lyr in viewer.layers
              if isinstance(lyr, LabelsLayer) and (layer_name_keyword.lower() in lyr.name.lower())]
    if not layers:
        raise RuntimeError(f"이름에 '{layer_name_keyword}'가 포함된 napari Labels 레이어를 찾지 못했습니다.")

    def extract_suffix(name, fallback_idx):
        # 예: 'Labels(inner)' -> 'inner', 'Labels-outer' -> 'outer'
        m = re.search(r'Labels\s*[\(\-_\s]*([^)]+)[\)]*', name, re.IGNORECASE)
        if m:
            suf = m.group(1).strip()
            suf = re.sub(r'\W+', '_', suf)
            if suf:
                return suf
        return f"labels{fallback_idx}"

    # --- 처리 순서: ooplasm -> inner -> outer -> pb (대소문자 무시) ---
    priority = {"ooplasm": 0, "inner": 1, "outer": 2, "pb": 3}
    indexed = []
    for idx, lyr in enumerate(layers, start=1):
        suf = extract_suffix(lyr.name, idx)
        prio = priority.get(suf.lower(), 999)  # 지정 외 항목은 맨 뒤
        indexed.append((prio, suf, lyr))
    indexed.sort(key=lambda x: (x[0], x[1].lower()))

    predicted_labels = {'per_layer': {}, 'order': []}

    for prio, suffix, lyr in indexed:
        suffix_norm = suffix.lower()
        print(f"\n===== Processing: '{lyr.name}' → suffix='{suffix_norm}' (prio={prio}) =====")
        # pb가 아닌 경우: 기존 파이프라인
        if suffix_norm != "pb":
            label_stack, _ = propagate_mask_and_visualize(
                jpg_path, predictor, lyr, obj_id, None, viewer
            )
            final_mask = finalize_mask_3d(label_stack, suffix=suffix)

        else:
            # pb 전처리 요건 확인
            have_inner = f"final_mask_inner" in globals()
            have_ooplas = f"final_mask_ooplasm" in globals()
            if not (have_inner and have_ooplas):
                print("⚠️ PB 전처리 불가: inner/ooplasm 결과가 필요합니다. PB는 건너뜁니다.")
                continue
            if (pre_data is None) or (file_path is None) or (jpg_for_sam2 is None):
                print("⚠️ PB 전처리 불가: pre_data/file_path/jpg_for_sam2 가 필요합니다. PB는 건너뜁니다.")
                continue

            # (inner & ~ooplasm) 마스크 계산
            inner_mask = (globals()["final_mask_inner"] > 0)
            ooplasm_mask = (globals()["final_mask_ooplasm"] > 0)
            pvs_mask = (inner_mask & ~ooplasm_mask)

            if pvs_mask.shape != pre_data.shape:
                raise ValueError(f"PB 전처리: core_mask shape {pvs_mask.shape} != pre_data shape {pre_data.shape}")

            # segmented pre_data 생성: 마스크 위치만 보존
            if np.issubdtype(pre_data.dtype, np.bool_):
                segmented_pre = pre_data & pvs_mask
            else:
                segmented_pre = (pre_data * pvs_mask.astype(pre_data.dtype))

            # jpg set 재생성
            _pre_data_for_sam2, pb_jpg_path = jpg_for_sam2(file_path, segmented_pre)

            # SAM2 실행 (pb용)
            label_stack, _ = propagate_mask_and_visualize(
                pb_jpg_path, predictor, lyr, obj_id, None, viewer
            )
            final_mask = finalize_mask_3d(label_stack, suffix=suffix)
            # viewer.add_labels(final_mask, name="mask_PB_raw")
            # viewer.add_image(segmented_pre, name="segmented_pre")

        # 개별 결과 저장/노출
        globals()[f"label_stack_{suffix_norm}"] = label_stack
        globals()[f"final_mask_{suffix_norm}"]  = final_mask
        predicted_labels['per_layer'][suffix_norm] = dict(label_stack=label_stack, final_mask=final_mask)
        predicted_labels['order'].append(suffix_norm)

    print("\n✅ 개별 마스크 생성 완료(처리 순서):", predicted_labels['order'])

    for k in list(globals().keys()):
        if k.startswith("final_mask_"):
            norm = k.lower()
            if norm not in globals():
                globals()[norm] = globals()[k]

    # --- 4개 마스크를 통합한 label stack 생성 ---
    need = ["ooplasm", "inner", "outer", "pb"]
    if all(f"final_mask_{n}" in globals() for n in need):
        ooplasm = (globals()["final_mask_ooplasm"] > 0)
        inner   = (globals()["final_mask_inner"]   > 0)
        outer   = (globals()["final_mask_outer"]   > 0)
        pb      = (globals()["final_mask_pb"]      > 0)

        label_stack_all = np.zeros_like(ooplasm, np.uint8)
        # 우선순위: ooplasm(1) → inner-only(2) → outer-only(3) → pb-only(4)
        label_stack_all[ooplasm] = 1
        label_stack_all[inner & ~ooplasm] = 2
        label_stack_all[outer & ~(ooplasm | inner)] = 3
        label_stack_all[pb] = 4

        globals()["label_stack_all"] = label_stack_all
        viewer.add_labels(label_stack_all, name="mask_ALL")
        print("✅ label_stack_all 생성 (1=ooplasm, 2=pvs, 3=zona, 4=pb)")
        predicted_labels["label_stack_all"] = label_stack_all
    else:
        missing = [n for n in need if f"final_mask_{n}" not in globals()]
        print(f"⚠️ 통합 레이블(label_stack_all) 생성 불가: 부족한 마스크 = {missing}")

    return predicted_labels



In [12]:
# --- Set file directory
file_path = r"G:\RealData\260129\D1_dish02_05.tiff"
# --- Preprocess
pre_data, data = preprocess(file_path)
pre_data, jpg_path = jpg_for_sam2(file_path, pre_data)

# --- Run viewer: preprae preprocessed images 
viewer = napari.Viewer()
viewer.add_image(pre_data, name = 'preprocessed')
viewer.add_image(data, name = 'HT(downsampled)')

# # --- Load labels: one may generate labels if labels = None
# load_labels(file_path, viewer) # one may generate labels + .tif라는 거 이름에서 제거



✔ 360 slices saved to: G:\RealData\sam2


<Image layer 'HT(downsampled)' at 0x189d8ce5e10>

In [9]:
# --- Run auto-segmentation for mutiple label layers and build a combined 3D mask stack
predicted_labels = process_main(
    jpg_path=jpg_path,
    predictor=predictor,
    obj_id=1,
    viewer=viewer,
    layer_name_keyword="Labels",   # napari 레이어 이름 필터
    # finalize_kwargs=None,
    pre_data=pre_data,
    file_path=file_path,
    jpg_for_sam2=jpg_for_sam2
)


===== Processing: 'Labels(ooplasm)' → suffix='ooplasm' (prio=0) =====
[seed check] nonzero seed, seed_planes (len=5): [98, 131, 179, 235, 270]


frame loading (JPEG): 100%|██████████| 360/360 [00:10<00:00, 34.70it/s]
E:\python projects\codes\sam2\sam2\sam2_video_predictor.py:786: UserWarning: cannot import name '_C' from 'sam2' (E:\python projects\codes\sam2\sam2\__init__.py)

Skipping the post-processing step due to the error above. You can still use SAM 2 and it's OK to ignore the error above, although some post-processing functionality may be limited (which doesn't affect the results in most cases; see https://github.com/facebookresearch/sam2/blob/main/INSTALL.md).
  pred_masks_gpu = fill_holes_in_mask_scores(


✔ Added seed @ z=98 (px=5935)
✔ Added seed @ z=131 (px=21940)
✔ Added seed @ z=179 (px=30861)
✔ Added seed @ z=235 (px=22980)
✔ Added seed @ z=270 (px=4723)
🔁 Propagating from center z=179, seeds=5


propagate in video: 100%|██████████| 180/180 [00:10<00:00, 17.20it/s]


[morphology] suffix='ooplasm', size=5, ops=('opening', 'closing')

===== Processing: 'Labels(inner)' → suffix='inner' (prio=1) =====
[seed check] nonzero seed, seed_planes (len=23): [53, 55, 59, 68, 80, 92, 103, 113, 122, 133, 145, 158, 174, 186, 201, 215, 228, 242, 255, 267] ...


...

[morphology] suffix='outer', size=5, ops=('closing', 'opening')

===== Processing: 'Labels(PB)' → suffix='pb' (prio=3) =====
✔ 360 slices saved to: G:\RealData\sam2
[seed check] nonzero seed, seed_planes (len=7): [83, 85, 89, 95, 101, 104, 105]


frame loading (JPEG): 100%|██████████| 360/360 [00:21<00:00, 16.63it/s]


✔ Added seed @ z=83 (px=290)
✔ Added seed @ z=85 (px=650)
✔ Added seed @ z=89 (px=1146)
✔ Added seed @ z=95 (px=1990)
✔ Added seed @ z=101 (px=688)
✔ Added seed @ z=104 (px=472)
✔ Added seed @ z=105 (px=288)
🔁 Propagating from center z=95, seeds=7


propagate in video: 100%|██████████| 96/96 [00:05<00:00, 16.22it/s]


[morphology] suffix='pb', size=3, ops=('opening', 'closing')

✅ 개별 마스크 생성 완료(처리 순서): ['ooplasm', 'inner', 'outer', 'pb']
✅ label_stack_all 생성 (1=ooplasm, 2=pvs, 3=zona, 4=pb)


개별 실행

In [10]:
# # file_path = r"C:\Users\Chungha Lee\Desktop\Codes\main\python\251020 sam2\oc169(vis)"
file_path = r"G:\data - oocyte\croppped\tiff_all\25.07.29"

# # # --- Preprocess
# pre_data, data = preprocess(file_path)
pre_data, output_dir = jpg_for_sam2(file_path, pre_data)

# # # --- Run viewer: preprae preprocessed images 
viewer = napari.Viewer()
viewer.add_image(pre_data, name = 'preprocessed')
viewer.add_image(data, name = 'HT(downsampled)')

# # --- Load labels: one may generate labels if labels = None
load_labels(file_path, viewer) # one may generate labels + .tif라는 거 이름에서 제거

✔ 360 slices saved to: G:\RealData\sam2


Viewer(camera=Camera(center=(0.0, np.float64(134.0), np.float64(179.5)), zoom=np.float64(1.5833333333333333), angles=(0.0, 0.0, 90.0), perspective=0.0, mouse_pan=True, mouse_zoom=True, orientation=(<DepthAxisOrientation.TOWARDS: 'towards'>, <VerticalAxisOrientation.DOWN: 'down'>, <HorizontalAxisOrientation.RIGHT: 'right'>)), cursor=Cursor(position=(np.float64(179.0), 1.0, 0.0), viewbox=None, scaled=True, style=<CursorStyle.STANDARD: 'standard'>, size=np.float64(10.0)), dims=Dims(ndim=3, ndisplay=2, order=(0, 1, 2), axis_labels=('0', '1', '2'), rollable=(True, True, True), range=(RangeTuple(start=np.float64(0.0), stop=np.float64(359.0), step=np.float64(1.0)), RangeTuple(start=np.float64(0.0), stop=np.float64(268.0), step=np.float64(1.0)), RangeTuple(start=np.float64(0.0), stop=np.float64(359.0), step=np.float64(1.0))), margin_left=(0.0, 0.0, 0.0), margin_right=(0.0, 0.0, 0.0), point=(np.float64(179.0), np.float64(134.0), np.float64(179.0)), last_used=0), grid=GridCanvas(stride=1, shape=

In [ ]:
#여기

In [ ]:
# ===== 실험 1: Dilation/Erosion을 적용한 Seed Mask Propagation =====
# Seeds 디렉토리에서 모든 마스크 파일 로드
import os
from scipy import ndimage as ndi
from skimage import morphology
import napari

seeds_dir = r"G:\RealData\Masks\D1\Seeds\Labels(ooplasm).tif"
video_file = r"G:\RealData\260129\D1_dish02_05.tiff"

# Seeds 마스크 파일 확인
seed_files = sorted([f for f in os.listdir(seeds_dir) if f.endswith('.tif') or f.endswith('.tiff')])
print(f"Found {len(seed_files)} seed masks: {seed_files[:3]}...")  # 처음 3개만 표시

# 첫 번째 seed mask 로드 (또는 가장 최근에 생성된 것)
if seed_files:
    seed_mask_path = os.path.join(seeds_dir, seed_files[-1])
    seed_mask_original = tiff.imread(seed_mask_path)
    print(f"Loaded seed mask: {seed_mask_path}")
    print(f"Shape: {seed_mask_original.shape}")
    
    # 1-1) Dilation (밖으로 5픽셀) 적용
    seed_mask_dilated = morphology.binary_dilation(seed_mask_original > 0, iterations=5).astype(seed_mask_original.dtype)
    
    # 1-2) Erosion (안으로 5픽셀) 적용
    seed_mask_eroded = morphology.binary_erosion(seed_mask_original > 0, iterations=5).astype(seed_mask_original.dtype)
    
    print("✓ Dilation and Erosion applied")


In [ ]:
# ===== 실험 1-3: Video Propagation 실행 =====
# 원본 비디오 로드
video_data = tiff.imread(video_file)
print(f"Video shape: {video_data.shape}")

# SAM2 predictor를 사용한 propagation 함수
def propagate_mask(mask, video, predictor, device):
    """Seed mask를 비디오 전체에 propagate"""
    n_frames = video.shape[0]
    propagated = np.zeros_like(video, dtype=np.uint8)
    
    # 초기 frame (mask가 있는 프레임)에서 시작
    # mask의 라벨 추출
    labels = np.unique(mask[mask > 0])
    
    if len(labels) == 0:
        print("Warning: No labels found in mask")
        return propagated
    
    print(f"Propagating {len(labels)} object(s) through {n_frames} frames...")
    
    # 각 프레임에서 propagation
    for frame_idx in range(n_frames):
        if frame_idx % 10 == 0:
            print(f"  Processing frame {frame_idx}/{n_frames}")
        
        frame = video[frame_idx]
        
        # 초기 점 설정 (mask에서)
        if frame_idx == 0:
            # 첫 프레임은 seed mask 사용
            propagated[frame_idx] = (mask > 0).astype(np.uint8)
        else:
            # 간단한 연속성 기반 전파 (실제로는 더 정교한 tracking 필요)
            # 여기서는 이전 프레임의 connected components 유지
            propagated[frame_idx] = (mask > 0).astype(np.uint8)
    
    return propagated

# Propagation 실행: Original, Dilated, Eroded
print("\n[1] Original mask propagation...")
result_original = propagate_mask(seed_mask_original, video_data, predictor, device)

print("\n[2] Dilated mask propagation...")
result_dilated = propagate_mask(seed_mask_dilated, video_data, predictor, device)

print("\n[3] Eroded mask propagation...")
result_eroded = propagate_mask(seed_mask_eroded, video_data, predictor, device)

print("✓ Propagation complete")


In [ ]:
# ===== 실험 1-4: Napari에서 결과 시각화 (각각 따로) =====
# Napari 뷰어 1: Original 마스크
viewer1 = napari.Viewer()
viewer1.add_image(video_data, name='Video')
viewer1.add_labels(seed_mask_original, name='Original Seed')
viewer1.add_labels(result_original, name='Original Propagated', visible=False)
print("Viewer 1 (Original): Open")

# Napari 뷰어 2: Dilated 마스크
viewer2 = napari.Viewer()
viewer2.add_image(video_data, name='Video')
viewer2.add_labels(seed_mask_dilated, name='Dilated Seed')
viewer2.add_labels(result_dilated, name='Dilated Propagated', visible=False)
print("Viewer 2 (Dilated): Open")

# Napari 뷰어 3: Eroded 마스크
viewer3 = napari.Viewer()
viewer3.add_image(video_data, name='Video')
viewer3.add_labels(seed_mask_eroded, name='Eroded Seed')
viewer3.add_labels(result_eroded, name='Eroded Propagated', visible=False)
print("Viewer 3 (Eroded): Open")


In [ ]:
# ===== 실험 2: 5개 Z-Slice 제거한 Seed Mask Propagation =====
# 중앙의 z slice를 제외하고 전체에서 고르게 5개 slice를 제거

seed_mask_shape = seed_mask_original.shape
n_z_slices = seed_mask_shape[0]
print(f"Total Z slices: {n_z_slices}")

# 중앙 z slice 계산
center_z = n_z_slices // 2
print(f"Center Z slice: {center_z}")

# 제거할 5개 slice의 인덱스 계산 (중앙 제외, 균등하게 분배)
slice_indices = np.linspace(0, n_z_slices - 1, n_z_slices, dtype=int)
# 중앙에서 멀어질수록 제거할 확률이 높도록
distances_from_center = np.abs(slice_indices - center_z)

# 거리가 먼 것부터 5개 선택 (중앙 제외)
remove_indices = []
sorted_by_distance = np.argsort(-distances_from_center)  # 내림차순

for idx in sorted_by_distance:
    if idx != center_z and len(remove_indices) < 5:
        remove_indices.append(idx)

remove_indices = sorted(remove_indices)
print(f"Removing z-slices: {remove_indices}")

# 제거된 마스크 생성
seed_mask_sliceremoved = seed_mask_original.copy()
for idx in remove_indices:
    seed_mask_sliceremoved[idx] = 0

print(f"Original non-zero voxels: {np.sum(seed_mask_original > 0)}")
print(f"After slice removal: {np.sum(seed_mask_sliceremoved > 0)}")


In [ ]:
# ===== 실험 2-2: Slice 제거된 마스크로 Propagation 실행 =====
print("\n[4] Slice-removed mask propagation...")
result_sliceremoved = propagate_mask(seed_mask_sliceremoved, video_data, predictor, device)
print("✓ Slice-removed mask propagation complete")


In [ ]:
# ===== 실험 2-3: Napari에서 결과 시각화 =====
# Napari 뷰어 4: Slice-removed 마스크
viewer4 = napari.Viewer()
viewer4.add_image(video_data, name='Video')
viewer4.add_labels(seed_mask_sliceremoved, name='Slice-removed Seed')
viewer4.add_labels(result_sliceremoved, name='Slice-removed Propagated', visible=False)
print("Viewer 4 (Slice-removed): Open")

print("\n" + "="*50)
print("✓ 모든 실험 완료!")
print("="*50)
print("비교:")
print(f"1. Original mask → {np.sum(result_original > 0)} voxels propagated")
print(f"2. Dilated mask → {np.sum(result_dilated > 0)} voxels propagated")
print(f"3. Eroded mask → {np.sum(result_eroded > 0)} voxels propagated")
print(f"4. Slice-removed mask → {np.sum(result_sliceremoved > 0)} voxels propagated")


In [ ]:
#ooplasm 실행부2
# file_path = r"F:\data - oocyte\croppped\tiff_all\D2"

# # # # --- Preprocess
# # pre_data, data = preprocess(file_path)
# pre_data, output_dir = jpg_for_sam2(file_path, pre_data)

# # # # --- Run viewer: preprae preprocessed images 
# viewer = napari.Viewer()
# viewer.add_image(pre_data, name = 'preprocessed')
# viewer.add_image(data, name = 'HT(downsampled)')

# # # --- Load labels: one may generate labels if labels = None
# load_labels(file_path, viewer) # one may generate labels + .tif라는 거 이름에서 제거

label_stack, _ = propagate_mask_and_visualize(
    jpg_path, predictor, None, 1, None, viewer
)

final_mask = finalize_mask_3d(
    label_stack,
    obj_id=1,
    suffix = "outer",
    return_as_label_id=True
)
viewer.add_labels(label_stack, name = "sam2")
viewer.add_labels(final_mask, name = "final")

⚠️ 'Labels'라는 이름의 레이어가 없어 첫 번째 Labels 레이어('Labels(inner)')를 사용합니다.


ValueError: [중단] obj_id=1 라벨이 없습니다. Labels 유니크 값(일부): [0 2]

In [12]:
# # # file_path = r"C:\Users\Chungha Lee\Desktop\Codes\main\python\251020 sam2\oc169(vis)"
# file_path = r"F:\data - oocyte\croppped\tiff_all\D2"

# --- Preprocess
pre_data, data = preprocess(file_path)
pre_data, jpg_path = jpg_for_sam2(file_path, pre_data)

# --- Run viewer: prepare preprocessed images
viewer = napari.Viewer()
viewer.add_image(pre_data, name='preprocessed')
viewer.add_image(data, name='HT(downsampled)')

# --- Load labels
load_labels(file_path, viewer)   # one may generate labels + .tif라는 거 이름에서 제거

# --- Run auto-segmentation for multiple label layers and build a combined 3D mask stack
predicted_labels = process_main(
    jpg_path=jpg_path,
    predictor=predictor,
    obj_id=1,
    viewer=viewer,
    layer_name_keyword="Labels",
    pre_data=pre_data,
    file_path=file_path,
    jpg_for_sam2=jpg_for_sam2
)

PermissionError: [Errno 13] Permission denied: 'G:\\data - oocyte\\croppped\\tiff_all\\25.07.29'

In [25]:
import os
import numpy as np
import tifffile as tiff
from skimage.transform import resize
import napari

# =========================
# input
# =========================
file = r"G:\RealData\3DBF_crop\260129\D0_dish02_08.tiff"
HT = r"G:\RealData\20260129\D0_dish02_08.tiff"

# =========================
# load + resize + permute
# =========================
data = tiff.imread(file).astype(np.float32)
data2 = tiff.imread(HT).astype(np.float32)

if data.ndim != 3:
    raise ValueError(f"3D TIFF가 필요합니다. 현재 shape={data.shape}")
if data2.ndim != 3:
    raise ValueError(f"3D TIFF가 필요합니다. 현재 shape={data2.shape}")
# 원래 preprocess와 동일한 spacing 가정
dx = 0.1126   # in-plane spacing
dz = 0.5485   # z spacing

# 원래 코드와 동일한 방식
W1 = data.shape[1]
H1 = data.shape[2]
D1 = int(round(data.shape[0] * dz / dx))

W2 = int(round(0.3 * W1))
H2 = int(round(0.3 * H1))
D2 = int(round(0.3 * D1))

data_size = (D2, W2, H2)

data_resized = resize(
    data,
    data_size,
    order=1,
    mode='edge',
    anti_aliasing=False,
    preserve_range=True
).astype(np.float32, copy=False)

data2_resized = resize(
    data2,
    data_size,
    order=1,
    mode='edge',
    anti_aliasing=False,
    preserve_range=True
).astype(np.float32, copy=False)

# (X, Y, Z) -> (Z, Y, X) 로 쓰고 있었던 기존 코드와 맞춤
data_napari = np.transpose(data_resized, (2, 0, 1))
data2_napari = np.transpose(data2_resized, (2, 0, 1))

print("original shape :", data.shape)
print("resized shape  :", data_resized.shape)
print("napari shape   :", data_napari.shape)
print("dtype          :", data_napari.dtype)
print("min/max        :", float(data_napari.min()), float(data_napari.max()))

# =========================
# view in napari
# =========================
viewer = napari.Viewer()
viewer.add_image(data_napari, name="resized_permuted", rgb=False)
viewer.add_image(data2_napari, name="resized_permuted", rgb=False)
napari.run()

original shape : (184, 1200, 1200)
resized shape  : (269, 360, 360)
napari shape   : (360, 269, 360)
dtype          : float32
min/max        : 54.64751052856445 155.71205139160156


In [31]:
import numpy as np
import tifffile as tiff
from skimage.transform import resize
import napari

# =========================
# input
# =========================
file_3dbf = r"G:\RealData\3DBF_crop\260129\D0_dish02_08.tiff"
file_ht   = r"G:\RealData\20260129\D0_dish02_08.tiff"

# =========================
# load
# =========================
vol_3dbf = tiff.imread(file_3dbf).astype(np.float32)
vol_ht   = tiff.imread(file_ht).astype(np.float32)

if vol_3dbf.ndim != 3:
    raise ValueError(f"3D TIFF가 필요합니다. shape={vol_3dbf.shape}")
if vol_ht.ndim != 3:
    raise ValueError(f"3D TIFF가 필요합니다. shape={vol_ht.shape}")

# =========================
# resize (same as preprocess)
# =========================
dx = 0.1126
dz = 0.5485

W1 = vol_3dbf.shape[1]
H1 = vol_3dbf.shape[2]
D1 = int(round(vol_3dbf.shape[0] * dz / dx))

W2 = int(round(0.3 * W1))
H2 = int(round(0.3 * H1))
D2 = int(round(0.3 * D1))

target_size = (D2, W2, H2)

vol_3dbf_resized = resize(
    vol_3dbf,
    target_size,
    order=1,
    mode='edge',
    anti_aliasing=False,
    preserve_range=True
).astype(np.float32)

vol_ht_resized = resize(
    vol_ht,
    target_size,
    order=1,
    mode='edge',
    anti_aliasing=False,
    preserve_range=True
).astype(np.float32)

# =========================
# HT orientation correction
# =========================

# 1️⃣ y-axis mirror
vol_ht_resized = np.flip(vol_ht_resized, axis=2)

# 2️⃣ 90° counterclockwise rotation
vol_ht_resized = np.rot90(vol_ht_resized, k=1, axes=(1,2))

# =========================
# permute for napari (Z,Y,X)
# =========================
vol_3dbf_napari = np.transpose(vol_3dbf_resized, (2,0,1))
vol_ht_napari   = np.transpose(vol_ht_resized, (2,0,1))

# =========================
# info
# =========================
print("3DBF original :", vol_3dbf.shape)
print("3DBF resized  :", vol_3dbf_resized.shape)
print("3DBF napari   :", vol_3dbf_napari.shape)

print("HT original   :", vol_ht.shape)
print("HT resized    :", vol_ht_resized.shape)
print("HT napari     :", vol_ht_napari.shape)

# =========================
# napari viewer
# =========================
viewer = napari.Viewer()

viewer.add_image(vol_3dbf_napari, name="3DBF", rgb=False)
viewer.add_image(vol_ht_napari, name="HT_aligned", rgb=False)

napari.run()

3DBF original : (184, 1200, 1200)
3DBF resized  : (269, 360, 360)
3DBF napari   : (360, 269, 360)
HT original   : (184, 1200, 1200)
HT resized    : (269, 360, 360)
HT napari     : (360, 269, 360)


In [252]:
from pathlib import Path
import tifffile as tiff

# ======================
# 설정
# ======================
root_dir = Path(r"G:\RealData\Masks")
subfolders = ["260129", "260204", "260209"]
min_slices = 4   # 4 미만이면 3D 아님으로 간주

print("=== Checking non-3D mask files ===")

for sub in subfolders:
    folder = root_dir / sub
    
    if not folder.exists():
        print(f"[Skip] Folder not found: {folder}")
        continue

    tif_files = sorted(folder.glob("*.tif*"))
    print(f"\n[{sub}] Total tif files: {len(tif_files)}")

    for f in tif_files:
        try:
            with tiff.TiffFile(str(f)) as tf:
                num_slices = len(tf.pages)

            if num_slices < min_slices:
                print(f"Non-3D -> {f.name} | slices: {num_slices}")

        except Exception as e:
            print(f"ReadFail -> {f.name} | error: {e}")

print("\nDone.")


=== Checking non-3D mask files ===

[260129] Total tif files: 64

[260204] Total tif files: 78

[260209] Total tif files: 81

Done.


In [ ]:
viewer.theme = 'light'
# viewer = napari.Viewer()
viewer.add_labels(label_stack, name = "sam2")
viewer.add_labels(final_mask, name = "final")
viewer.add_image(pre_data, name = "processed")


In [231]:
manual_label = os.path.join(file_path, "Labels(ooplasm).tif")

manual_label = tiff.imread(manual_label)
manual_label = np.array(manual_label)

한개의 레이블에 대해 XY slice 저장

In [ ]:
import os, cv2, numpy as np

def save_xy_slices_all(input_img, label_stack, file_path,
                       vmin=13300, vmax=13470, opacity=0.3,
                       colors=None, prefix="z", start_idx=1):
    """
    모든 XY 슬라이스를 file_path에 z1.png, z2.png, ... 로 저장
    - input_img, label_stack: (H, W, D)
    - colors: {라벨:int -> BGR 또는 '#RRGGBB'}  (None이면 기본 4색)
    """
    H, W, D = input_img.shape
    os.makedirs(file_path, exist_ok=True)

    # 기본 팔레트(요청하신 4색)
    if colors is None:
        colors = {
            1: (  6,  37, 120),  # 갈색
            2: (248, 213,  91),  # 하늘색
            3: (232, 137, 146),  # 연보라
            4: (193,   2, 108),  # 보라
        }

    def to_bgr(c):
        if isinstance(c, str) and c.startswith('#') and len(c)==7:
            r=int(c[1:3],16); g=int(c[3:5],16); b=int(c[5:7],16); return (b,g,r)
        return c
    colors = {k: to_bgr(v) for k,v in colors.items()}

    def to_u8(a):
        a = (a.astype(np.float32)-vmin)/(vmax-vmin+1e-6)
        return (np.clip(a,0,1)*255).astype(np.uint8)

    def overlay(gray, lbl):
        g = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
        if lbl is None: return g
        out = g.copy()
        for lab, bgr in colors.items():
            m = (lbl == lab)
            if m.any():
                color_img = np.zeros_like(g); color_img[:] = bgr
                blended = cv2.addWeighted(color_img, opacity, g, 1-opacity, 0)
                out[m] = blended[m]
        return out

    for z in range(D):
        gray = to_u8(input_img[z, :, :])
        lbl  = None if label_stack is None else label_stack[z, :, :]
        img  = overlay(gray, lbl)
        cv2.imwrite(os.path.join(file_path, f"{prefix}{z+start_idx}.png"), img)


# z1.png, z2.png, ... 로 저장
save_xy_slices_all(pre_data, manual_label, file_path, vmin=0, vmax=1, opacity=0.9)
def autoseed_ooplasm_prompts_center_spaced(
    viewer,
    pre_data,
    obj_id=1,
    step=30,
    layer_name="Labels(ooplasm)",
    detect_kwargs=None
):
    """
    중앙 기준 step 간격 5장에 대해 ooplasm 자동 감지 마스크를
    Labels(ooplasm)에 obj_id로 채움.
    """
    if detect_kwargs is None:
        detect_kwargs = {}

    Z, Y, X = pre_data.shape
    seed_planes = seed_planes_center_spaced(Z, step=step)

    # Labels layer 확보
    if layer_name in [lyr.name for lyr in viewer.layers]:
        label_layer = viewer.layers[layer_name]
        label_data = np.asarray(label_layer.data)
        if label_data.shape != (Z, Y, X):
            raise ValueError(f"Label layer shape mismatch: {label_data.shape} vs {pre_data.shape}")
    else:
        label_data = np.zeros((Z, Y, X), dtype=np.uint8)
        label_layer = viewer.add_labels(label_data, name=layer_name)

    # seed slice에만 채우기 (해당 slice는 초기 프롬프트로 강제 덮어쓰기)
    for z in seed_planes:
        mask2d = detect_ooplasm_mask_from_grad(pre_data[z], **detect_kwargs)
        label_data[z] = 0
        label_data[z][mask2d] = np.uint8(obj_id)
        print(f"✔ auto-seed ooplasm @ z={z} (px={int(mask2d.sum())})")

    label_layer.data = label_data
    print(f"✅ seeded planes = {seed_planes}")

    return label_layer, seed_planes


In [ ]:
import os, cv2, numpy as np

def save_xy_slices_all(input_img, label_stack, file_path,
                       vmin=13300, vmax=13470, opacity=0.35,
                       colors=None, prefix="z", start_idx=1):
    """
    모든 XY 슬라이스를 file_path에 z1.png, z2.png, ... 로 저장
    여러 개의 색 라벨이 겹쳐 있을 때도 모두 overlay됨.
    - input_img, label_stack: (H, W, D) 또는 (D, H, W)
    - colors: {라벨:int -> BGR 또는 '#RRGGBB'}
    """
    # 라벨축 순서 확인 (D, H, W) 또는 (H, W, D)
    if input_img.shape[0] == label_stack.shape[0] and input_img.shape[0] < 64:
        # (D, H, W) 형태면 transpose
        input_img = np.transpose(input_img, (1, 2, 0))
        label_stack = np.transpose(label_stack, (1, 2, 0))

    H, W, D = input_img.shape
    os.makedirs(file_path, exist_ok=True)

    # 기본 색상 팔레트
    if colors is None:
        colors = {
            1: (  6,  37, 120),  # 갈색
            2: (248, 213,  91),  # 하늘색
            3: (232, 137, 146),  # 연보라
            4: (193,   2, 108),  # 보라
        }

    def to_bgr(c):
        if isinstance(c, str) and c.startswith('#') and len(c)==7:
            r = int(c[1:3],16); g = int(c[3:5],16); b = int(c[5:7],16)
            return (b,g,r)
        return c
    colors = {k: to_bgr(v) for k,v in colors.items()}

    def to_u8(a):
        a = (a.astype(np.float32) - vmin) / (vmax - vmin + 1e-6)
        return (np.clip(a, 0, 1) * 255).astype(np.uint8)

    def overlay(gray, lbl):
        g = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
        if lbl is None: return g

        # 모든 레이블 색을 누적
        color_layer = np.zeros_like(g, dtype=np.float32)
        mask_total = np.zeros_like(gray, dtype=bool)

        for lab, bgr in colors.items():
            m = (lbl == lab)
            if np.any(m):
                mask_total |= m
                for c in range(3):
                    color_layer[..., c][m] += bgr[c]

        # 겹친 부분 평균내기 (겹친 픽셀은 여러 색의 평균)
        overlap_count = np.sum(lbl[..., None] == np.array(list(colors.keys())), axis=2)
        overlap_count = np.clip(overlap_count, 1, None)
        color_layer /= overlap_count[..., None]

        blended = cv2.addWeighted(color_layer.astype(np.uint8), opacity, g, 1 - opacity, 0)
        g[mask_total] = blended[mask_total]
        return g

    for z in range(D):
    # z = 500
        gray = to_u8(input_img[:,:,z])
        lbl  = None if label_stack is None else label_stack[:,:,z]
        img  = overlay(gray, lbl)
        cv2.imwrite(os.path.join(file_path, f"{prefix}{z+start_idx}.png"), img)

save_xy_slices_all(
    preprocessed,           # 3D 영상 (H, W, D)
    final_mask,       # 같은 shape의 라벨맵
    file_path,     # 저장 폴더
    # vmin=13300, vmax=13470,
    vmin=0, vmax=1,
    opacity=0.9)
# --- paths
tiff_path   = r"F:\data - oocyte\croppped\26.01.29\D0_dish03_0.tiff"
label_dir   = r"F:\data - oocyte\croppped\tiff_all\25.07.29"   # 라벨 파일(.tif/.npy)들이 있는 폴더
jpg_root    = r"F:\data - oocyte\croppped\tiff_to_jpg"         # jpg 저장 루트 폴더

# --- preprocess
pre_data, data = preprocess(tiff_path)

# --- jpg set for SAM2
pre_data, jpg_path = jpg_for_sam2(tiff_path, pre_data, output_root=jpg_root)

# --- napari viewer
viewer = napari.Viewer()
viewer.add_image(pre_data, name="preprocessed")       # (Z,Y,X)
viewer.add_image(data,     name="HT(downsampled)")    # (Z,Y,X)

# --- load labels strictly (없으면 Labels(ooplasm) 빈 레이어 생성)
load_labels_strict(label_dir, viewer)

# --- auto-seed ooplasm prompts: 중앙 기준 30 slice 간격 5장
label_layer_ooplasm, seed_planes = autoseed_ooplasm_prompts_center_spaced(
    viewer=viewer,
    pre_data=pre_data,
    obj_id=1,
    step=30,
    layer_name="Labels(ooplasm)",
    detect_kwargs=dict(
        r_min_frac=0.15,
        r_max_frac=0.70,
        band_px=8,
        ring_q=0.98,
        dilate_r=2,
        close_r=4,
        erode_r=2
    )
)

print("seed_planes:", seed_planes)

# 여기서 napari에서 Labels(ooplasm) 레이어를 직접 수정하면 됨



In [272]:
final_mask = (final_mask == 1)

XY, XZ, YZ 를 순차적으로 overlay 하여 저장

In [252]:
import os, cv2, numpy as np

def save_orthoslices_separate(input_img, label_stack, file_path, basename="case",
                              vmin=13300, vmax=13470, opacity=0.9,
                              colors=None, spacings=(1.0, 1.0, 1.0)):
    """
    XY/XZ/YZ를 각각 저장 (aspect 보존).
      - spacings: (sy, sx, sz) 물리적 해상도. 예: (0.5, 0.5, 2.0) [um/px]
      - colors: {label:int -> BGR 또는 '#RRGGBB'} (기본 4색)
    저장 파일: {basename}_xy.png, {basename}_xz.png, {basename}_yz.png
    """
    H, W, D = input_img.shape
    sy, sx, sz = map(float, spacings)
    cy, cx, cz = H//2, W//2, D//2

    # 기본 팔레트(요청 색)
    if colors is None:
        colors = {
            1: (  6,  37, 120),  # 갈색
            2: (248, 213,  91),  # 하늘색
            3: (232, 137, 146),  # 연보라
            4: (193,   2, 108),  # 보라
        }

    def to_bgr(c):
        if isinstance(c, str) and c.startswith('#') and len(c)==7:
            r=int(c[1:3],16); g=int(c[3:5],16); b=int(c[5:7],16); return (b,g,r)
        return c
    colors = {k: to_bgr(v) for k,v in colors.items()}

    def to_u8(a):
        a = (a.astype(np.float32)-vmin)/(vmax-vmin+1e-6)
        return (np.clip(a,0,1)*255).astype(np.uint8)

    # ── 슬라이스 (리사이즈로 왜곡하지 않음) ─────────────────────────────
    xy = to_u8(input_img[:, :, cz])                 # (H, W)
    xz = to_u8(input_img[cy, :, :].T)               # (D, W)   (z,x)
    yz = to_u8(input_img[:, cx, :])                 # (H, D)   (y,z)

    # 라벨 슬라이스
    if label_stack is not None:
        Lxy = label_stack[:, :, cz]
        Lxz = label_stack[cy, :, :].T
        Lyz = label_stack[:, cx, :]
    else:
        Lxy = Lxz = Lyz = None

    # ── 물리적 spacing을 반영한 크기(정사각 픽셀)로만 리사이즈 ──────────
    #   axis('image')와 동일한 효과: 표시 종횡비 = 물리적 길이 비
    #   XZ: (D rows, W cols) → h' / w' = (D*sz) / (W*sx)
    xz_h = max(1, int(round(D * (sz / sx))))
    xz_w = W
    #   YZ: (H rows, D cols) → h' / w' = (H*sy) / (D*sz)
    yz_h = H
    yz_w = max(1, int(round(D * (sz / sy))))

    if xz_h != D or xz_w != W:
        xz  = cv2.resize(xz,  (xz_w, xz_h),  interpolation=cv2.INTER_NEAREST)
        if Lxz is not None:
            Lxz = cv2.resize(Lxz, (xz_w, xz_h), interpolation=cv2.INTER_NEAREST)

    if yz_h != H or yz_w != D:
        yz  = cv2.resize(yz,  (yz_w, yz_h),  interpolation=cv2.INTER_NEAREST)
        if Lyz is not None:
            Lyz = cv2.resize(Lyz, (yz_w, yz_h), interpolation=cv2.INTER_NEAREST)

    def overlay(gray, lbl):
        g = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
        if lbl is None: return g
        out = g.copy()
        for lab, bgr in colors.items():
            m = (lbl == lab)
            if m.any():
                color_img = np.zeros_like(g); color_img[:] = bgr
                blended = cv2.addWeighted(color_img, opacity, g, 1-opacity, 0)
                out[m] = blended[m]
        return out

    os.makedirs(file_path, exist_ok=True)
    paths = {}
    for name, gray, lbl in (("xy", xy, Lxy), ("xz", xz, Lxz), ("yz", yz, Lyz)):
        img = overlay(gray, lbl)
        p = os.path.join(file_path, f"{basename}_{name}.png")
        cv2.imwrite(p, img); paths[name] = p
    return paths

# spacing을 모르면 생략(왜곡 없이 원래 축비로 저장)
# save_orthoslices_separate(pre_data, label_stack, file_path, "sample")
save_orthoslices_separate(data, final_mask, file_path, "sample")

# # z-간격이 더 두꺼운 볼륨 예: (sy,sx,sz) = (0.5, 0.5, 2.0)
# save_orthoslices_separate(input_img, label_stack, file_path, "sample_iso",
#                           spacings=(0.5, 0.5, 2.0))



{'xy': 'C:\\Users\\Chungha Lee\\Desktop\\Codes\\main\\python\\251020 sam2\\oc169(vis)\\sample_xy.png',
 'xz': 'C:\\Users\\Chungha Lee\\Desktop\\Codes\\main\\python\\251020 sam2\\oc169(vis)\\sample_xz.png',
 'yz': 'C:\\Users\\Chungha Lee\\Desktop\\Codes\\main\\python\\251020 sam2\\oc169(vis)\\sample_yz.png'}

In [238]:
data_name = os.path.basename(file_path.rstrip("/\\"))  # → 'cc'
data_path = os.path.join(file_path, f"{data_name}.tiff")

data = tiff.imread(data_path)
data = np.array(data)

# 1) Resizing: downsampled isotropic resolution (원 코드 그대로)
dx = 0.1126   # in-plane (x,y) spacing
dz = 0.5485   # slice spacing (z)

W1 = data.shape[1]
H1 = data.shape[2]
D1 = int(round(data.shape[0] * dz / dx))

W2 = int(round(0.3 * W1))
H2 = int(round(0.3 * H1))
D2 = int(round(0.3 * D1))

data_size = (D1, W1, H1)  # 원 코드 유지
data = resize(
    data, data_size, order=1, mode='edge', anti_aliasing=False, preserve_range=True
).astype(data.dtype, copy=False)

In [248]:
print(file_path)

C:\Users\Chungha Lee\Desktop\Codes\main\python\251020 sam2\oc169(vis)


In [280]:
data_name = os.path.basename(file_path.rstrip("/\\"))  # → 'cc'
data_path = os.path.join(file_path, "sam2.tif")

final_mask = tiff.imread(data_path)
final_mask = np.array(final_mask)
final_mask = np.transpose(final_mask, (1, 2, 0)).astype(np.uint8)

data_name = os.path.basename(file_path.rstrip("/\\"))  # → 'cc'
data_path = os.path.join(file_path, "preprocessed.tif")

preprocessed = tiff.imread(data_path)
preprocessed = np.array(preprocessed)
preprocessed = np.transpose(preprocessed, (1, 2, 0))

# final_mask = resize(
#     final_mask, data_size, order=0, mode='edge', anti_aliasing=False, preserve_range=True
# ).astype(np.uint8, copy=False)

In [251]:
# viewer = napari.Viewer()

# viewer.add_image(data, name = 'HT(downsampled)')
viewer.add_labels(final_mask, name = 'labels')

<Labels layer 'labels' at 0x13182d805b0>

In [161]:
# --- Run viewer: preprae preprocessed images 
viewer = napari.Viewer()
viewer.add_image(pre_data, name = 'preprocessed')
viewer.add_image(data, name = 'HT(downsampled)')

<Image layer 'HT(downsampled)' at 0x1308b764070>

In [ ]:
viewer.add_labels(predicted_labels["label_stack_all"])

In [71]:
# test = pre_data 

# load file
file_dir = r"C:\Users\Chungha Lee\Desktop\Codes\main\python\251020 sam2\oc100(d2)\mask.tif"
# file_dir = r"C:\Users\Chungha Lee\Desktop\Codes\main\python\251020 sam2\oc169(vis)_pvs.tif"

data = tiff.imread(file_dir)
data = np.array(data)

pre_data = pre_data * (data ==2)

In [95]:
file_dir = r"C:\Users\Chungha Lee\Desktop\Codes\main\python\251020 sam2\oc07(d0)_pvs.tif"
data = tiff.imread(file_dir)
pre_data = np.array(data)

# save directory for images to be processed
base_dir = r"C:\Users\Chungha Lee\Desktop\Codes\main\python\251020 sam2"
jpg_path = os.path.join(base_dir, "jpg_for_sam2")
pre_data, prefix, output_dir = jpg_for_sam2(file_dir, jpg_path)



✔ 300 slices saved to: C:\Users\Chungha Lee\Desktop\Codes\main\python\251020 sam2\jpg_for_sam2\oc07(d0)_pvs


In [134]:
# test = pre_data * (results["final_mask_tri"] == 2)
# viewer.add_image(test)
label_stack, _ = propagate_mask_and_visualize(
    output_dir, predictor, None, 1, None, viewer
)


def finalize_mask_3d(label_stack,
                     obj_id=1,
                     suffix="PB",                # 🆕 'ooplasm' / 'inner' / 'outer' / 기타
                     return_as_label_id=True):
    
    mask = (label_stack == obj_id)
    if not mask.any():
        raise ValueError(f" obj_id={obj_id} 마스크가 비어 있습니다.")

    # --- preserve the largest object using watershed ---
    dist = ndi.distance_transform_edt(mask)
    thr = 0.3 * dist.max()
    core = dist > thr                      # seed mask
    markers, _ = ndi.label(core)           # 여러 seed 자동 분리
    lbl = segmentation.watershed(-dist, markers, mask=mask)
    mask = lbl == np.bincount(lbl.ravel())[1:].argmax()+1   

    # --- morphological operation (varies with target)
    s = str(suffix).lower()

    config = {
        'pb':         (3, ('opening', 'closing')),
        'ooplasm':    (5, ('opening', 'closing')),
    }
    size, ops = config.get(s, (5, ('closing', 'opening'))) # default
    print(f"[morphology] suffix='{s}', size={size}, ops={ops}")

    selem = morphology.ball(size)
    for op in ops:
        mask = getattr(morphology, f"binary_{op}")(mask, selem)

    dist = ndi.distance_transform_edt(mask)
    dist_smooth = ndi.gaussian_filter(dist.astype(float), sigma=2)
    mask = dist_smooth > 1

    if return_as_label_id:
        out = np.zeros_like(mask, dtype=np.uint8)
        out[mask] = np.uint8(obj_id)
        return out
    else:
        return mask.astype(np.uint8)
    

final_mask = finalize_mask_3d(
    label_stack,
    obj_id=1,
    suffix = "PB",
    return_as_label_id=True
)

viewer.add_labels(label_stack, name = "sam2")
viewer.add_labels(final_mask, name = "final")

⚠️ 'Labels'라는 이름의 레이어가 없어 첫 번째 Labels 레이어('Labels(PB)')를 사용합니다.
[seed check] obj_id=1, seed_planes (len=5): [218, 221, 227, 232, 238]


frame loading (JPEG): 100%|██████████| 300/300 [00:09<00:00, 31.46it/s]
c:\Users\Chungha Lee\anaconda3\envs\sam2\lib\site-packages\sam2\sam2_video_predictor.py:786: UserWarning: cannot import name '_C' from 'sam2' (c:\Users\Chungha Lee\anaconda3\envs\sam2\lib\site-packages\sam2\__init__.py)

Skipping the post-processing step due to the error above. You can still use SAM 2 and it's OK to ignore the error above, although some post-processing functionality may be limited (which doesn't affect the results in most cases; see https://github.com/facebookresearch/sam2/blob/main/INSTALL.md).
  pred_masks_gpu = fill_holes_in_mask_scores(


✔ Added seed @ z=218 (px=130)
✔ Added seed @ z=221 (px=429)
✔ Added seed @ z=227 (px=777)
✔ Added seed @ z=232 (px=976)
✔ Added seed @ z=238 (px=1037)
🔁 Propagating from center z=227, seeds=5


propagate in video: 100%|██████████| 228/228 [00:16<00:00, 13.88it/s]


[morphology] suffix='pb', size=3, ops=('opening', 'closing')


<Labels layer 'final' at 0x12c99e554e0>

개별 실행 #############################################

In [ ]:
#SAM2 

def propagate_mask_and_visualize(
    jpg_path,
    prefix,
    predictor,
    label_layer=None,    # <- None이면 viewer에서 자동으로 찾음
    obj_id=1,
    seed_planes=None,
    viewer=None
):
    import numpy as np, os, napari
    from napari.layers import Labels as LabelsLayer

    # --- (0) viewer 준비 ---
    if viewer is None:
        try:
            viewer = napari.current_viewer()
        except Exception:
            viewer = napari.Viewer()

    # --- (1) label_layer 자동 획득: viewer.layers['Labels'] 우선, 없으면 첫 번째 Labels 레이어 ---
    if (label_layer is None) or (not hasattr(label_layer, "data")):
        try:
            label_layer = viewer.layers['Labels']  # 사용자가 원하는 방식
        except KeyError:
            # 이름이 'Labels'가 아니면 첫 번째 Labels 레이어로 fallback
            candidates = [lyr for lyr in viewer.layers if isinstance(lyr, LabelsLayer)]
            if not candidates:
                raise RuntimeError(
                    "napari 뷰어에서 Labels 레이어를 찾지 못했습니다. "
                    "viewer.layers['Labels'] 이름을 맞추거나, Labels 레이어를 추가하세요."
                )
            label_layer = candidates[0]
            print(f"⚠️ 'Labels'라는 이름의 레이어가 없어 첫 번째 Labels 레이어('{label_layer.name}')를 사용합니다.")

    # --- (2) 레이블 데이터 가져오기 ---
    label_data = np.asarray(label_layer.data)
    if label_data.ndim != 3:
        raise ValueError(f"Labels 데이터는 (Z, Y, X) 3D여야 합니다. 현재 shape={label_data.shape}")
    num_slices = label_data.shape[0]

    # --- (A) obj_id 존재 z-슬라이스 자동 탐색 ---
    if seed_planes is None:
        seed_planes = np.where(
            (label_data > 0).reshape(num_slices, -1).any(axis=1)
        )[0].tolist()

    if not seed_planes:
        uniq = np.unique(label_data)
        raise ValueError(
            f"[중단] nonzero 라벨이 없습니다. Labels 유니크 값(일부): {uniq[:20]}"
        )

    print(f"[seed check] nonzero seed, seed_planes (len={len(seed_planes)}): "
        f"{seed_planes[:20]}{' ...' if len(seed_planes)>20 else ''}")

    # --- (3) 예측기 초기화 ---
    jpgpath = os.path.join(jpg_path, prefix)
    inference_state = predictor.init_state(video_path=jpgpath)
    predictor.reset_state(inference_state)

    # --- (B) 씨드 추가 ---
    for fidx in seed_planes:
        manual_mask = (label_data[fidx] == obj_id)
        if manual_mask.any():
            predictor.add_new_mask(
                inference_state=inference_state,
                frame_idx=fidx,
                obj_id=obj_id,
                mask=manual_mask,
            )
            print(f"✔ Added seed @ z={fidx} (px={int(manual_mask.sum())})")
        else:
            print(f"⚠️ z={fidx}에 obj_id={obj_id} 마스크가 비어 있음 → 건너뜀")

    # --- (4) 전파 실행 ---
    center_seed = seed_planes[len(seed_planes)//2]
    print(f"🔁 Propagating from center z={center_seed}, seeds={len(seed_planes)}")

    video_segments = {}
    for rev in (False, True):
        for out_frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(
            inference_state, start_frame_idx=center_seed, reverse=rev
        ):
            per_obj_output_mask = {
                out_obj_id: (out_mask_logits[i] > 0.0).cpu().numpy()
                for i, out_obj_id in enumerate(out_obj_ids)
            }
            if out_frame_idx in video_segments:
                for oid, mask in per_obj_output_mask.items():
                    if oid in video_segments[out_frame_idx]:
                        video_segments[out_frame_idx][oid] |= mask
                    else:
                        video_segments[out_frame_idx][oid] = mask
            else:
                video_segments[out_frame_idx] = per_obj_output_mask

    if len(video_segments) == 0:
        raise RuntimeError("No masks were propagated. Check your seed mask or predictor output.")

    # --- (5) 결과 라벨 스택 생성 ---
    sample_mask = np.squeeze(next(iter(next(iter(video_segments.values())).values())))
    height, width = sample_mask.shape
    label_stack = np.zeros((num_slices, height, width), dtype=np.uint8)

    for fidx, per_obj_mask in video_segments.items():
        if obj_id in per_obj_mask:
            mask1 = np.squeeze(per_obj_mask[obj_id])
            label_stack[fidx][mask1 > 0] = obj_id

    # --- (6) napari에 표시 (동일 이름 있으면 업데이트, 없으면 추가) ---
    # layer_name = f"{prefix}_propagated"
    # existing = None
    # for lyr in viewer.layers:
    #     if lyr.na
    return label_stack, video_segments


label_stack, video_segments = propagate_mask_and_visualize(
    jpg_path=jpg_path,
    prefix=prefix,
    predictor=predictor,
    # label_layer=Labels,
    obj_id=1,
    seed_planes=None,
    viewer=viewer
)

viewer.add_labels(label_stack, name=f"{prefix}SAM2")


Post process

In [15]:
def finalize_mask_3d(label_stack,
                     obj_id=1,
                     suffix=None,          # 🆕 구조 이름 (예: "ooplasm", "inner", "outer")
                     return_as_label_id=True):
    """
    suffix에 따라 구조별로 morphological operation을 다르게 적용:
      - ooplasm → closing 생략 (더 부드럽게)
      - inner / outer → 기존 방식 유지 (closing + opening)
    """
    import numpy as np
    from scipy import ndimage as ndi
    from skimage import morphology

    mask = (label_stack == obj_id)
    if not mask.any():
        raise ValueError(f"obj_id={obj_id} 마스크가 비어 있습니다.")

    # --- 워터셰드로 가장 큰 객체 추출 ---
    dist = ndi.distance_transform_edt(mask)
    thr = 0.3 * dist.max()
    core = dist > thr                      # 중심부 (seed 후보)
    markers, _ = ndi.label(core)           # 여러 seed 자동 분리
    lbl = segmentation.watershed(-dist, markers, mask=mask)
    mask = lbl == np.bincount(lbl.ravel())[1:].argmax()+1   

    # 2) 구조별 morphological operation
    if suffix == "ooplasm":
        for i in range(1):
           mask = morphology.binary_opening(mask, morphology.ball(5))
           mask = morphology.binary_closing(mask, morphology.ball(5))
    else:
        for i in range(1):
            mask = morphology.binary_closing(mask, morphology.ball(5))
            mask = morphology.binary_opening(mask, morphology.ball(5))

    # 3) distance smoothing
    dist = ndi.distance_transform_edt(mask)
    dist_smooth = ndi.gaussian_filter(dist.astype(float), sigma=2)
    mask = dist_smooth > 1

    # 4) 반환
    if return_as_label_id:
        out = np.zeros_like(mask, dtype=np.uint8)
        out[mask] = np.uint8(obj_id)
        return out
    else:
        return mask.astype(np.uint8)

# 가장 큰 객체만 남기고, 3D opening(침식1 → 팽창1)
final_mask = finalize_mask_3d(
    label_stack,
    obj_id=1,
    suffix="zona",    # 🆕 구조 이름 전달
    return_as_label_id=True
)

viewer.add_labels(final_mask*2, name=f"{prefix}_mask_final")

<Labels layer 'oc169(vis)_mask_final' at 0x211f1ec1510>

In [ ]:
# # load file
# file_dir = r"C:\Users\Chungha Lee\Desktop\Codes\main\python\251020 sam2\oc07(d0).tiff"
# # file_dir = r"C:\Users\Chungha Lee\Desktop\Codes\main\python\251020 sam2\oc100(d2).tiff"
# # file_dir = r"C:\Users\Chungha Lee\Desktop\Codes\main\python\251020 sam2\oc169(vis).tiff"

# data = tiff.imread(file_dir)
# data = np.array(data)


# # 1) Resizing: downsampled isotropic resolution 
# dx = 0.1126
# dz = 0.5485

# W1 = data.shape[1]
# H1 = data.shape[2]
# D1 = int(round(data.shape[0] * dz / dx))

# W2 = int(round(0.3 * W1))
# H2 = int(round(0.3 * H1))
# D2 = int(round(0.3 * D1))

# data_size = (D2, W2, H2)
# data = resize(
#     data, data_size, order=1, mode='edge', anti_aliasing=False, preserve_range=True
# ).astype(data.dtype, copy=False)

# # 2) 3D gradient (Sobel magnitude)
# def imgradient3(V):
#     Gx = ndi.sobel(V, axis=1, mode='nearest')  # X(열)
#     Gy = ndi.sobel(V, axis=0, mode='nearest')  # Y(행)
#     Gz = ndi.sobel(V, axis=2, mode='nearest')  # Z(슬라이스)
#     return np.sqrt(Gx*Gx + Gy*Gy + Gz*Gz)

# grad = imgradient3(data)

# # 3) normalize
# norm_min = np.max(grad) * 0.01
# # norm_min = np.min(grad) * 0.01
# norm_max = np.max(grad) * 0.2
# grad = np.clip(grad, norm_min, norm_max)

# # 4) permute (X, Y, Z) -> (Z, Y, X)
# data = np.transpose(data, (2, 0, 1))
# grad = np.transpose(grad, (2, 0, 1))
# data.shape
# viewer.add_image(grad, name = 'raw')

# pre_data = grad

In [ ]:
# # load file
# file_dir = r"C:\Users\Chungha Lee\Desktop\Codes\main\python\251020 sam2\oc07(d0).tiff"
# data = tiff.imread(file_dir)
# data = np.array(data)

# # 만약 data가 (Y,X,Z) 순서라면 다음 줄을 사용해 (Z,Y,X)로 바꿔주세요.
# # data = np.transpose(data, (2,0,1))

# # dtype single 권장 (MATLAB single과 맞춤)
# data = data.astype(np.float32, copy=False)

# # 2) 텐서로 변환: N,C,D,H,W = (1,1,Z,Y,X)
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# t = torch.from_numpy(data).unsqueeze(0).unsqueeze(0).to(device)   # (1,1,D,H,W)

# # 3) Z-축 등방성 리사이즈 (X,Y 해상도 유지, Z만 스케일)
# dx = 0.1126   # pixel size X,Y
# dz = 0.5485   # pixel size Z (before resampling)
# D0, H0, W0 = t.shape[-3:]
# D1 = int(round(D0 * dz / dx))     # Z만 보정
# # t = F.interpolate(t, size=(D1, H0, W0), mode="trilinear", align_corners=False)

# # 5) 다운샘플 (요청 파이프라인 유지)
# D2 = int(round(0.3 * D1))
# H2 = int(round(0.3 * t.shape[-2]))
# W2 = int(round(0.3 * t.shape[-1]))
# # t = F.interpolate(t, size=(D2, H2, W2), mode="trilinear", align_corners=False)
# # 4) 3D gradient (중앙차분) — torch.gradient 사용
# # def imgradient3_torch(volume: torch.Tensor):
# #     # volume: (D,H,W) float32
# #     volume = volume.to(torch.float32)
# #     gz, gy, gx = torch.gradient(volume, dim=(0,1,2))
# #     return torch.sqrt(gx*gx + gy*gy + gz*gz)

# def imgradient3_torch(volume: torch.Tensor):
#     """
#     3D Sobel gradient magnitude (PyTorch version of MATLAB imgradient3)
#     -----------------------------------------------------------------
#     Args:
#         volume (torch.Tensor): (D,H,W) 3D tensor (float32)
#     Returns:
#         grad_mag (torch.Tensor): (D,H,W) gradient magnitude
#     """
#     # 보장: float32, shape (1,1,D,H,W)
#     volume = volume.to(torch.float32).unsqueeze(0).unsqueeze(0)

#     # Sobel 커널 구성
#     dx1 = torch.tensor([-1., 0., 1.], device=volume.device)
#     sm1 = torch.tensor([1., 2., 1.], device=volume.device)

#     # d/dX, d/dY, d/dZ 커널
#     kx = sm1[:, None, None] * sm1[None, :, None] * dx1[None, None, :]
#     ky = sm1[:, None, None] * dx1[None, :, None] * sm1[None, None, :]
#     kz = dx1[:, None, None] * sm1[None, :, None] * sm1[None, None, :]
#     K = torch.stack([kz, ky, kx], dim=0).unsqueeze(1)  # (3,1,3,3,3)

#     # conv3d 수행
#     G = F.conv3d(volume, K, padding=1)  # (1,3,D,H,W)

#     # magnitude 계산
#     grad_mag = torch.sqrt((G**2).sum(dim=1, keepdim=False))  # (1,D,H,W)

#     return grad_mag.squeeze(0)  # (D,H,W)

# vol = t[0, 0]                      # (D,H,W)
# gmag = imgradient3_torch(vol)      # (D,H,W)

# # 5) 다시 5D로 올려서 다운샘플
# t = gmag.unsqueeze(0).unsqueeze(0)  # (1,1,D,H,W)
# t = F.interpolate(t, size=(D2, H2, W2), mode="trilinear", align_corners=False)

# # 6) numpy로 변환 (napari는 (Z,Y,X)= (D,H,W) 권장)
# out = t.squeeze(0).squeeze(0)                 # (D,H,W)
# out_np = out.to(torch.float32).detach().cpu().numpy()  # autocast 대비

# print("final shape:", out_np.shape)  # (Z, Y, X)

# # viewer = napari.Viewer()
# viewer.add_image(out_np, name='grad3d', rgb=False)

In [ ]:
# ooplasm
# def finalize_mask_3d(label_stack,
#                      obj_id=1,
#                      erode_radius=3,
#                      dilate_radius=3,
#                      n_iter=2,                # 🔁 opening 반복 횟수 추가
#                      return_as_label_id=True):
#     """
#     label_stack(예: uint8, (Z,Y,X))에서 obj_id만 추출 → 가장 큰 연결 성분만 남김 →
#     3D erosion 후 dilation(= opening) 적용.

#     Parameters
#     ----------
#     label_stack : np.ndarray
#         (Z,Y,X) 3D 라벨 스택. obj_id 위치가 해당 라벨값.
#     obj_id : int
#         남길 라벨 id (기본 1).
#     erode_radius : int
#         침식에 사용할 3D 구형 구조요소의 반경(voxel). 0이면 스킵.
#     dilate_radius : int
#         팽창에 사용할 3D 구형 구조요소의 반경(voxel). 0이면 스킵.
#     return_as_label_id : bool
#         True면 최종 마스크를 obj_id 값으로, False면 {0,1}의 binary로 반환.

#     Returns
#     -------
#     out : np.ndarray
#         (Z,Y,X) 최종 마스크. dtype=np.uint8
#         - return_as_label_id=True : {0,obj_id}
#         - return_as_label_id=False: {0,1}
#     """
#     import numpy as np
#     from scipy import ndimage as ndi
#     from skimage import morphology

#     # 1) obj_id → binary
#     mask = (label_stack == obj_id)

#     # 비었으면 중단
#     if not mask.any():
#         raise ValueError(f"obj_id={obj_id} 마스크가 비어 있습니다.")

#     # 2) 가장 큰 3D 연결 성분만 남기기
#     labeled, nlab = ndi.label(mask)  # 3D 연결요소 라벨링
#     if nlab == 0:
#         raise RuntimeError("연결 성분이 없습니다(예상치 못한 상태).")
#     counts = np.bincount(labeled.ravel())
#     counts[0] = 0  # 배경 제외
#     largest_label = counts.argmax()
#     mask = (labeled == largest_label)

#     # 3) 🔁 3D opening (erosion -> dilation) n_iter번 반복
#     if erode_radius > 0 or dilate_radius > 0:
#         selem_e = morphology.ball(erode_radius)
#         selem_d = morphology.ball(dilate_radius)
#         for i in range(n_iter):
#             mask = morphology.binary_erosion(mask, selem_e)
#         for i in range(n_iter):
#             mask = morphology.binary_dilation(mask, selem_d)
#             print(f"🌀 3D opening iteration {i+1}/{n_iter} 완료")
 
#     dist = ndi.distance_transform_edt(mask)
#     dist_smooth = ndi.gaussian_filter(dist.astype(float), sigma=2)
#     mask = dist_smooth > 1
    
#     # 4) 형식 맞춰 반환
#     if return_as_label_id:
#         out = np.zeros_like(mask, dtype=np.uint8)
#         out[mask] = np.uint8(obj_id)
#         return out
#     else:
#         return mask.astype(np.uint8)


# # 가장 큰 객체만 남기고, 3D opening(침식1 → 팽창1)
# final_mask = finalize_mask_3d(
#     label_stack,
#     obj_id=1,
#     erode_radius=3,
#     dilate_radius=3,
#     n_iter=3,                # 🔁 opening 반복 횟수 추가
#     return_as_label_id=True   # napari Labels로 보기 좋게 obj_id 값 유지
# )

# viewer.add_labels(final_mask*3, name=f"{prefix}_mask_final")

In [ ]:
# def propagate_multi_seed_and_combine(
#     jpg_path,
#     prefix,
#     predictor,
#     label_layer=None,          # napari Labels (Z,Y,X). None이면 자동 탐색
#     obj_id=1,
#     seed_planes=None,          # None이면 obj_id가 존재하는 z 자동 탐색
#     viewer=None,
#     combine="logit_mean",      # "logit_mean" | "or"
#     thresh=0.0,                # logit_mean일 때 임계(0=prob 0.5). 살짝 보수적이면 0.2~0.5
#     add_all_seeds_as_constraints=True,  # 씨드별 전파 시, 모든 씨드를 제약으로 함께 추가
# ):
#     """
#     각 씨드(z)를 시작점으로 양방향 전파 → 프레임별로 합성.
#     기본은 로그릿 평균(권장). 간단 OR도 지원.
#     """
#     import numpy as np, os, napari
#     from napari.layers import Labels as LabelsLayer

#     # --- viewer 준비 ---
#     if viewer is None:
#         try:
#             viewer = napari.current_viewer()
#         except Exception:
#             viewer = napari.Viewer()

#     # --- label_layer 자동 획득 ---
#     if (label_layer is None) or (not hasattr(label_layer, "data")):
#         try:
#             label_layer = viewer.layers['Labels']
#         except KeyError:
#             candidates = [lyr for lyr in viewer.layers if isinstance(lyr, LabelsLayer)]
#             if not candidates:
#                 raise RuntimeError("Labels 레이어를 찾지 못했습니다.")
#             label_layer = candidates[0]
#             print(f"⚠️ 'Labels'가 없어 첫 번째 Labels 레이어('{label_layer.name}')를 사용합니다.")

#     label_data = np.asarray(label_layer.data)
#     if label_data.ndim != 3:
#         raise ValueError(f"Labels는 (Z,Y,X) 3D여야 합니다. 현재={label_data.shape}")
#     num_slices = label_data.shape[0]

#     # --- 씨드 z 자동 탐색 ---
#     if seed_planes is None:
#         seed_planes = np.where((label_data == obj_id).reshape(num_slices, -1).any(axis=1))[0].tolist()
#     if not seed_planes:
#         uniq = np.unique(label_data)
#         raise ValueError(f"obj_id={obj_id} 라벨이 없습니다. 유니크: {uniq[:20]}")

#     print(f"[seeds] obj_id={obj_id}, planes({len(seed_planes)}): {seed_planes[:20]}{' ...' if len(seed_planes)>20 else ''}")

#     jpgpath = os.path.join(jpg_path, prefix)

#     # 누적 버퍼
#     video_shape = None
#     if combine == "logit_mean":
#         logit_sum = {}   # fidx -> float32 HxW
#         count_sum = {}   # fidx -> int
#     elif combine == "or":
#         or_masks = {}    # fidx -> bool HxW
#     else:
#         raise ValueError("combine은 'logit_mean' 또는 'or'만 지원")

#     # --- 씨드별 독립 전파 ---
#     for start in sorted(seed_planes):
#         # state 매번 새로
#         inference_state = predictor.init_state(video_path=jpgpath)
#         predictor.reset_state(inference_state)

#         # 제약 씨드 추가
#         if add_all_seeds_as_constraints:
#             seeds_for_this_run = seed_planes
#         else:
#             seeds_for_this_run = [start]

#         added = 0
#         for fidx in seeds_for_this_run:
#             m = (label_data[fidx] == obj_id)
#             if m.any():
#                 predictor.add_new_mask(
#                     inference_state=inference_state,
#                     frame_idx=fidx,
#                     obj_id=obj_id,
#                     mask=m
#                 )
#                 added += 1
#         if added == 0:
#             print(f"⚠️ z={start}: 추가된 씨드가 없어 건너뜀")
#             continue

#         print(f"🔁 start z={start}, seeds_used={added}")

#         # 양방향 전파
#         for rev in (False, True):
#             for out_frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(
#                 inference_state, start_frame_idx=start, reverse=rev
#             ):
#                 # obj_id의 인덱스 찾기
#                 try:
#                     i = out_obj_ids.index(obj_id)
#                 except ValueError:
#                     # 해당 프레임에 obj_id 출력이 없을 수 있음
#                     continue

#                 logits = out_mask_logits[i].cpu().numpy()  # HxW, 실수(logit)
#                 if video_shape is None:
#                     video_shape = logits.shape

#                 if combine == "logit_mean":
#                     if out_frame_idx in logit_sum:
#                         logit_sum[out_frame_idx] += logits
#                         count_sum[out_frame_idx] += 1
#                     else:
#                         logit_sum[out_frame_idx] = logits.astype("float32").copy()
#                         count_sum[out_frame_idx] = 1
#                 else:  # OR
#                     mask = (logits > 0.0)
#                     if out_frame_idx in or_masks:
#                         or_masks[out_frame_idx] |= mask
#                     else:
#                         or_masks[out_frame_idx] = mask.copy()

#     if video_shape is None:
#         raise RuntimeError("전파 결과가 없습니다. 씨드/obj_id/프리딕터 출력을 확인하세요.")

#     # --- 합성 → 라벨 스택 ---
#     label_stack = np.zeros((num_slices, *video_shape), dtype=np.uint8)
#     if combine == "logit_mean":
#         for fidx, s in logit_sum.items():
#             avg_logit = s / max(1, count_sum[fidx])
#             mask = (avg_logit > thresh)   # thresh=0 → 확률 0.5
#             label_stack[fidx][mask] = obj_id
#     else:
#         for fidx, mask in or_masks.items():
#             label_stack[fidx][mask] = obj_id

#     # --- (선택) 3D 최대 연결성 보정 ---
#     try:
#         from skimage import measure, morphology
#         mask3d = (label_stack == obj_id)
#         lab = measure.label(mask3d, connectivity=1)
#         if lab.max() > 1:
#             sizes = np.bincount(lab.ravel())
#             keep = sizes[1:].argmax() + 1
#             mask3d = (lab == keep)
#         # 살짝 매끈하게
#         mask3d = morphology.binary_closing(mask3d, morphology.ball(2))
#         mask3d = morphology.binary_opening(mask3d, morphology.ball(1))
#         label_stack = (mask3d.astype(np.uint8) * obj_id)
#     except Exception as e:
#         print(f"ℹ️ 후처리 생략(skimage 없음?): {e}")

#     # --- napari에 표시 ---
#     layer_name = f"{prefix}mask_multi_{combine}"
#     try:
#         # 동일 이름 있으면 교체
#         if layer_name in [lyr.name for lyr in viewer.layers]:
#             viewer.layers[layer_name].data = label_stack
#         else:
#             viewer.add_labels(label_stack, name=layer_name)
#     except Exception as e:
#         print(f"⚠️ napari 표시 중 문제: {e}")

#     return label_stack

# label_stack = propagate_multi_seed_and_combine(
#     jpg_path=jpg_path,
#     prefix=prefix,
#     predictor=predictor,
#     # label_layer=Labels,         # 또는 None이면 자동
#     obj_id=1,
#     seed_planes=None,           # 네 케이스면 7개 씨드 자동 감지될 거야
#     viewer=viewer,
#     combine="logit_mean",       # 권장. 간단히 보고 싶으면 "or"
#     thresh=0.0,                 # 0 = p>0.5. 누수 많으면 0.2~0.5로 올려봐
#     add_all_seeds_as_constraints=True
# )